# CS-781 Capstone — Phase 7: Prior Case Retrieval (PCR), round 3

> **Round 3 (two new submissions, about 15-25 minutes, no retraining):** attach the OUTPUT of your round-2 Phase-7 run (plus the Phase-1 dataset). The trained models S1c, S2 and S3c are loaded from it; Section 13 re-ranks with the citations and dates in the case headers (a query that cites a candidate's citation, a candidate whose CITATOR INFO lists the query, a candidate decided after the query) and writes `p7_R3a_B-cite_dev....zip` (S1c) and `p7_R3b_Ens3-cite_dev....zip` (S1c + S2 + S3c ensemble), copied to the top of the output folder.

> **Round 2 (earlier):** attach the OUTPUT of your first Phase-7 run (plus the Phase-1 dataset). It continues S1 (B, full fine-tuning) for 6 more epochs -> **S1c**, retrains the distilled student from S1c -> **S3c**, and trains C with partial fine-tuning (top 4 encoder layers) instead of IA³ -> **S4b**. About 1.5-2 hours. Without the first run's output, S1 and S3 are trained first (about 1 hour more).

This notebook fine-tunes **your own Phase-1 language models** (A, B, C, D) to rank prior cases for a query case and writes, for every test query, the **7** most relevant candidates from the test pool. One zip per experiment, short descriptive names (at most 64 characters), e.g. `p7_S1_B_FullFT_hybrid_dev0.3124.zip`, all in `cs781_phase7/submissions/`.

## Before you run it
1. **Settings (right panel):** Accelerator **GPU T4 x2**, Internet **on**.
2. **Add-ons → Secrets:** turn on the secret **`HF`**. The data is downloaded from `Exploration-Lab/CS-781-Capstone/Phase_7_PCR`.
3. **Add Input:** your Phase-1 dataset (the same one as before).
4. *Save Version → Save & Run All.* To continue later, attach this notebook's previous output too: finished experiments are loaded, not retrained.

## The approach
- **Bi-encoder:** query and candidate cases are encoded by the same fine-tuned Phase-1 model, independently. A case is cut into windows of ~340 words (512 tokens); up to 8 windows spread over the case are read; a window's vector is the learned average of all layers, averaged over its tokens; the case vector is the mean of its windows. Similarity: cosine of case vectors, or "maxsim" (each query window's best-matching candidate window, averaged).
- **Training:** contrastive (InfoNCE) with in-batch negatives and one **BM25 hard negative** per query; other relevant cases of the same query are never used as negatives.
- **Ranking:** alpha × model similarity + (1 − alpha) × **BM25** (both standardised per query). The similarity type and alpha (at least 0.2, so the model always takes part) are chosen on the dev queries. BM25 is a classical word-statistics formula, not a pretrained model.
- Every test query gets exactly 7 unique ids from the test pool, best first, never the query itself — checked before every zip is written.

| Submission | Model | Technique (Phase-7 list) |
|---|---|---|
| S1_B_full | B (GQA/RoPE) | **full fine-tuning** |
| S2_D_lorma | D (GPT, 400M tokens) | **LoRMA** (compulsory), query/value, rank 32 |
| S3_A4_kd | A (first 4 layers) | **knowledge distillation** from S1 (matches S1's similarity distribution in every batch) |
| S4_C_ia3 | C (encoder) | **IA³** (learned rescaling), word embeddings scaled up and mixed in |

## Rules this follows
Every experiment starts from your Phase-1 model; no other pretrained model, weights or API. Each split's queries are ranked only against that split's candidate pool.

## 1. Setup and Hugging Face login

In [ ]:
import os
import re
import sys
import json
import math
import time
import glob
import gc
import pickle
import copy
import random
import shutil
import zipfile
import hashlib
import warnings
from collections import Counter, defaultdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm

warnings.filterwarnings("ignore", category=FutureWarning)

HF_TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF")
except Exception as e:
    HF_TOKEN = os.environ.get("HF_TOKEN")
    print("Kaggle secret 'HF' not available (" + type(e).__name__ + "); using the HF_TOKEN environment variable if set.")
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    try:
        from huggingface_hub import login
        login(token=HF_TOKEN, add_to_git_credential=False)
        print("Hugging Face login successful.")
    except Exception as e:
        print("Hugging Face login failed:", repr(e))

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"
print("PyTorch:", torch.__version__, "| device:", DEVICE)
for i in range(torch.cuda.device_count()):
    print(f"GPU {i}:", torch.cuda.get_device_name(i))
if DEVICE.type == "cuda":
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
else:
    print("WARNING: no GPU. Use Kaggle 'GPU T4 x2' for real runs.")

## 2. Configuration and experiments

In [ ]:
SEED = 42
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
PHASE7_DIR = "Phase_7_PCR"
TOP_K = 7

INPUT_ROOT = os.environ.get("CS781_INPUT_ROOT", "/kaggle/input")
WORK_ROOT = os.environ.get("CS781_P7_WORK_ROOT",
                           "/kaggle/working/cs781_phase7" if os.path.isdir("/kaggle/working") else os.path.abspath("cs781_phase7"))
LOCAL_DATA_DIR = os.environ.get("CS781_P7_DATA_DIR", "")
SUBMISSION_DIR = os.path.join(WORK_ROOT, "submissions")
CACHE_DIR = os.path.join(WORK_ROOT, "cache")
for _d in (WORK_ROOT, SUBMISSION_DIR, CACHE_DIR):
    os.makedirs(_d, exist_ok=True)

PHASE1_MODELS = {
    "A": dict(architecture="A", pick="other"),
    "B": dict(architecture="B", pick="best"),
    "C": dict(architecture="C", pick="best"),
    "D": dict(architecture="A", pick="best"),
}

WINDOW_WORDS = 340
CHUNK_TOKENS = 512
MAX_CHUNKS = 8
EVAL_CHUNKS = 4
TRAIN_CHUNKS = 2

ALPHA_GRID = [round(a, 1) for a in np.arange(0.2, 1.01, 0.1)]
ALPHA_MIN = ALPHA_GRID[0]

DEFAULTS = dict(
    backbone="B", n_layers=None,
    technique="full",
    queries_per_batch=6,
    epochs=4, patience=2, max_pairs_per_epoch=3000,
    temperature=0.05,
    hard_negative_pool=50,
    lr_backbone=2e-5, lr_peft=5e-4, lr_head=1e-3,
    weight_decay=0.01, warmup_frac=0.06, grad_clip=1.0, dropout=0.1,
    lora_r=16, lora_alpha=32, lora_scaling="standard", peft_targets="qv", lorma_variant="plus",
    adapter_dim=64, partial_top_layers=4,
    embedding_state=False, pooled_state_norm=False, c_input_scale=1.0,
    teacher=None, kd_temperature=0.05, kd_weight=0.5,
    init_from=None,
    seed=SEED,
)

EXPERIMENTS = {
    "S1_B_full":   dict(backbone="B", technique="full"),
    "S2_D_lorma":  dict(backbone="D", technique="lorma", lora_r=32, lora_alpha=64),
    "S3_A4_kd":    dict(backbone="A", n_layers=4, technique="full", lr_backbone=5e-5,
                        teacher="S1_B_full"),
    "S4_C_ia3":    dict(backbone="C", technique="ia3", lr_peft=3e-3, embedding_state=True,
                        pooled_state_norm=True, c_input_scale="match"),
    "S1c_B_full":    dict(backbone="B", technique="full", init_from="S1_B_full", epochs=6, seed=SEED + 1),
    "S3c_A4_kd":     dict(backbone="A", n_layers=4, technique="full", lr_backbone=5e-5, teacher="S1c_B_full",
                          init_from="S3_A4_kd", epochs=4, seed=SEED + 1),
    "S4b_C_partial": dict(backbone="C", technique="partial", partial_top_layers=4, lr_backbone=5e-5,
                          embedding_state=True, pooled_state_norm=True, c_input_scale="match"),
    "X1_B_partial": dict(backbone="B", technique="partial"),
    "X2_D_ia3":     dict(backbone="D", technique="ia3", lr_peft=3e-3),
}
RUN_EXPERIMENTS = ["S1c_B_full", "S2_D_lorma", "S3c_A4_kd"]
if os.environ.get("CS781_P7_RUN"):
    RUN_EXPERIMENTS = [x for x in os.environ["CS781_P7_RUN"].split(",") if x]

TECH_TAGS = {"full": "FullFT", "partial": "PartFT", "lorma": "LoRMA", "ia3": "IA3", "lora": "LoRA", "adapter": "Adapter",
             "bitfit": "BitFit", "frozen": "Frozen"}

SMOKE_TEST = os.environ.get("CS781_P7_SMOKE", "0") == "1"
if SMOKE_TEST:
    WINDOW_WORDS, CHUNK_TOKENS, MAX_CHUNKS, EVAL_CHUNKS = 30, 48, 4, 2
    DEFAULTS.update(epochs=2, queries_per_batch=3, lora_r=4, lora_alpha=8, hard_negative_pool=5)
    for _cfg in EXPERIMENTS.values():
        if _cfg.get("n_layers"):
            _cfg["n_layers"] = 1
        _cfg.pop("lora_r", None)
        _cfg.pop("lora_alpha", None)
    print("*** SMOKE TEST MODE: tiny settings, results are meaningless ***")

def experiment_config(name):
    cfg = dict(DEFAULTS)
    if EXPERIMENTS[name].get("init_from"):
        cfg.update({k: v for k, v in EXPERIMENTS[EXPERIMENTS[name]["init_from"]].items() if k != "teacher"})
    cfg.update(EXPERIMENTS[name])
    cfg["name"] = name
    return cfg

def submission_name(cfg, dev_f1):
    """Short and descriptive (at most 64 characters), e.g. p7_S1_B_FullFT_hybrid_dev0.3124.zip"""
    model = cfg["backbone"] + (f"{cfg['n_layers']}L" if cfg["n_layers"] else "")
    tech = "KD" + cfg["teacher"].split("_")[0] if cfg["teacher"] else TECH_TAGS[cfg["technique"]]
    if cfg.get("init_from"):
        tech += f"-c{cfg['epochs']}ep"
    name = f"p7_{cfg['name'].split('_')[0]}_{model}_{tech}_hybrid_dev{dev_f1:.4f}.zip"
    assert len(name) <= 64, name
    return name

def restore_previous_outputs():
    """Copy a previous run of this notebook (attached with "Add Input") into the working folder, so finished
    experiments are re-used instead of trained again."""
    if not os.path.isdir(INPUT_ROOT):
        return
    copied = 0
    for dirpath, dirnames, _ in os.walk(INPUT_ROOT):
        if os.path.basename(dirpath) == "cs781_phase7":
            for sub, _, files in os.walk(dirpath):
                for name in files:
                    src = os.path.join(sub, name)
                    dst = os.path.join(WORK_ROOT, os.path.relpath(src, dirpath))
                    if not os.path.exists(dst):
                        os.makedirs(os.path.dirname(dst), exist_ok=True)
                        shutil.copy2(src, dst)
                        copied += 1
            dirnames[:] = []
        elif dirpath[len(INPUT_ROOT):].count(os.sep) >= 6:
            dirnames[:] = []
    if copied:
        print(f"Restored {copied} file(s) from a previous Phase-7 run.")

restore_previous_outputs()
print("Working folder:", WORK_ROOT)
print("Experiments to run:", RUN_EXPERIMENTS)
for _name in RUN_EXPERIMENTS:
    print(f"  {_name:14s} -> {submission_name(experiment_config(_name), 0.0).replace('dev0.0000', 'dev<score>')}")

## 3. Load the PCR data, case windows and BM25

In [ ]:
import scipy.sparse as sp

def load_json_file(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def write_json(obj, path):
    with open(path + ".tmp", "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)
    os.replace(path + ".tmp", path)

_HF_FILES = None

def phase7_file(name):
    """Local path of a Phase-7 file such as "train_candidates.json" (local folder, attached dataset, or Hugging Face)."""
    global _HF_FILES
    candidates = []
    if LOCAL_DATA_DIR:
        candidates += sorted(glob.glob(os.path.join(LOCAL_DATA_DIR, "**", name), recursive=True))
    if os.path.isdir(INPUT_ROOT):
        for depth in range(0, 4):
            candidates += sorted(glob.glob(os.path.join(INPUT_ROOT, "*", *(["*"] * depth), PHASE7_DIR, "**", name), recursive=True))
    for c in candidates:
        if os.path.isfile(c):
            return c
    from huggingface_hub import hf_hub_download, list_repo_files
    if _HF_FILES is None:
        _HF_FILES = list_repo_files(HF_DATASET_REPO, repo_type="dataset", token=HF_TOKEN)
    matches = sorted((f for f in _HF_FILES if f.startswith(PHASE7_DIR + "/") and f.split("/")[-1] == name), key=len)
    if not matches:
        raise FileNotFoundError(f"{name} not found in {HF_DATASET_REPO}/{PHASE7_DIR}")
    print(f"downloading {matches[0]} ...", flush=True)
    return hf_hub_download(repo_id=HF_DATASET_REPO, filename=matches[0], repo_type="dataset", token=HF_TOKEN)

_ID_KEYS = ("id", "query_id", "candidate_id", "case_id", "doc_id")
_TEXT_KEYS = ("text", "sentences", "document", "content")
_REL_KEYS = ("relevant_candidates", "relevant", "relevant_ids", "labels", "positives")

def _records(data):
    """A list of {...} records or a dict {id: {...} or id: text} -> list of dicts that all have "id"."""
    out = []
    if isinstance(data, dict):
        for k, v in data.items():
            rec = dict(v) if isinstance(v, dict) else {"text": v}
            if not any(key in rec for key in _ID_KEYS):
                rec["id"] = k
            out.append(rec)
    else:
        out = [dict(r) for r in data]
    return out

def _text(rec):
    for k in _TEXT_KEYS:
        if k in rec and rec[k] is not None:
            t = rec[k]
            return " ".join(" ".join(str(s) for s in t).split()) if isinstance(t, (list, tuple)) else " ".join(str(t).split())
    return ""

def _id(rec):
    for k in _ID_KEYS:
        if k in rec and rec[k] is not None:
            return str(rec[k])
    raise KeyError("record without an id: " + str(list(rec)[:6]))

def read_split(split):
    q_name, c_name = f"{split}.json", f"{split}_candidates.json"
    queries = []
    for r in _records(load_json_file(phase7_file(q_name))):
        rel = None
        for k in _REL_KEYS:
            if k in r and r[k] is not None:
                rel = r[k] if isinstance(r[k], (list, tuple)) else [r[k]]
                break
        queries.append({"id": _id(r), "text": _text(r), "rel": None if split == "test" else [str(x) for x in (rel or [])]})
    cands = _records(load_json_file(phase7_file(c_name)))
    pool = {"ids": [_id(r) for r in cands], "texts": [_text(r) for r in cands]}
    return queries, pool

QUERIES, POOLS = {}, {}
for _split in ("train", "dev", "test"):
    QUERIES[_split], POOLS[_split] = read_split(_split)
    _ids = POOLS[_split]["ids"]
    assert len(set(_ids)) == len(_ids), f"{_split}: duplicate candidate ids"
    POOLS[_split]["index"] = {cid: i for i, cid in enumerate(_ids)}
    _qids = [q["id"] for q in QUERIES[_split]]
    assert len(set(_qids)) == len(_qids), f"{_split}: duplicate query ids"
    if _split != "test":
        missing = sum(1 for q in QUERIES[_split] for c in q["rel"] if c not in POOLS[_split]["index"])
        for q in QUERIES[_split]:
            q["rel_idx"] = sorted({POOLS[_split]["index"][c] for c in q["rel"] if c in POOLS[_split]["index"] and c != q["id"]})
        if missing:
            print(f"{_split}: {missing} relevant ids are not in the candidate pool (ignored)")
assert len(POOLS["test"]["ids"]) > TOP_K, "the test pool is too small for 7 predictions"

print(f"{'split':6s} {'queries':>8s} {'pool':>7s} {'relevant/query':>15s} {'query words (median)':>21s} {'case words (median)':>20s}")
for _s in ("train", "dev", "test"):
    _r = f"{np.mean([len(q['rel_idx']) for q in QUERIES[_s]]):.2f}" if _s != "test" else "-"
    print(f"{_s:6s} {len(QUERIES[_s]):8,} {len(POOLS[_s]['ids']):7,} {_r:>15s} "
          f"{int(np.median([len(q['text'].split()) for q in QUERIES[_s]])):21,} "
          f"{int(np.median([len(t.split()) for t in POOLS[_s]['texts']])):20,}")
_self = sum(q["id"] in POOLS[s]["index"] for s in QUERIES for q in QUERIES[s])
print(f"Queries that also appear in their own pool: {_self} (a case is never returned for itself)")
print("Example query start:", QUERIES["train"][0]["text"][:300])

def windows(text, max_windows):
    """Consecutive windows of WINDOW_WORDS words; a longer case keeps max_windows of them spread evenly over the case."""
    w = text.split() or ["<empty>"]
    parts = [w[i:i + WINDOW_WORDS] for i in range(0, len(w), WINDOW_WORDS)]
    if len(parts) > 1 and len(parts[-1]) < WINDOW_WORDS // 4:
        parts[-2] = parts[-2] + parts[-1]
        parts = parts[:-1]
    if len(parts) > max_windows:
        keep = np.unique(np.round(np.linspace(0, len(parts) - 1, max_windows)).astype(int))
        parts = [parts[i] for i in keep]
    return [" ".join(p) for p in parts]

_WORD = re.compile(r"[a-z0-9]+")

def bm25_scores(split, k1=1.2, b=0.75):
    """[queries x pool] BM25 scores. Query terms are weighted by 1 + log(tf) (queries are whole judgments)."""
    path = os.path.join(CACHE_DIR, f"bm25_{split}_{len(QUERIES[split])}_{len(POOLS[split]['ids'])}.npy")
    if os.path.exists(path):
        return np.load(path)
    vocab = {}

    def counts(text):
        c = Counter(_WORD.findall(text.lower()))
        return {vocab.setdefault(t, len(vocab)): n for t, n in c.items()}

    rows, cols, vals, lengths = [], [], [], []
    for i, t in enumerate(tqdm(POOLS[split]["texts"], desc=f"BM25 index ({split})", leave=False)):
        c = counts(t)
        rows.append(np.full(len(c), i, dtype=np.int64))
        cols.append(np.fromiter(c.keys(), dtype=np.int64, count=len(c)))
        vals.append(np.fromiter(c.values(), dtype=np.float32, count=len(c)))
        lengths.append(sum(c.values()))
    n_docs, V = len(lengths), len(vocab)
    tf = sp.csr_matrix((np.concatenate(vals), (np.concatenate(rows), np.concatenate(cols))), shape=(n_docs, V))
    df = np.bincount(tf.indices, minlength=V).astype(np.float32)
    idf = np.log(1.0 + (n_docs - df + 0.5) / (df + 0.5)).astype(np.float32)
    lengths = np.asarray(lengths, dtype=np.float32)
    norm = k1 * (1 - b + b * lengths / max(1.0, lengths.mean()))
    tf = tf.tocoo()
    w = tf.data * (k1 + 1) / (tf.data + norm[tf.row]) * idf[tf.col]
    W = sp.csr_matrix((w, (tf.row, tf.col)), shape=(n_docs, V))
    qrows, qcols, qvals = [], [], []
    for i, q in enumerate(QUERIES[split]):
        for t, n in Counter(_WORD.findall(q["text"].lower())).items():
            j = vocab.get(t)
            if j is not None:
                qrows.append(i)
                qcols.append(j)
                qvals.append(1.0 + math.log(n))
    Q = sp.csr_matrix((np.asarray(qvals, dtype=np.float32), (qrows, qcols)), shape=(len(QUERIES[split]), V))
    scores = (Q @ W.T).toarray().astype(np.float32)
    np.save(path, scores)
    return scores

def exclude_self(split, scores):
    """A query is never ranked against itself (if it also appears in its pool)."""
    s = scores.copy()
    for i, q in enumerate(QUERIES[split]):
        j = POOLS[split]["index"].get(q["id"])
        if j is not None:
            s[i, j] = -np.inf
    return s

BM25 = {s: exclude_self(s, bm25_scores(s)) for s in ("train", "dev", "test")}

## 4. Phase-1 tokenizer

In [ ]:
SPECIAL_TOKENS = ["<pad>", "<unk>", "<s>", "</s>"]
PAD_ID = 0

class BPETokenizer:
    """Loads the byte-level BPE tokenizer saved by a Phase-1 notebook (tokenizer.json) and encodes text with the
    same merge rules. Only loading and encoding are needed here; the tokenizer is NOT retrained."""

    def __init__(self, directory):
        with open(os.path.join(directory, "tokenizer.json"), "r", encoding="utf-8") as f:
            data = json.load(f)
        self.token_to_id = {str(k): int(v) for k, v in data["token_to_id"].items()}
        self.merge_ranks = {tuple(pair): i for i, pair in enumerate(data["merges"])}
        self.unk_id = self.token_to_id["<unk>"]
        self.eow_id = self.token_to_id["</w>"]
        self.vocab_size = len(self.token_to_id)
        self._cache = {}
        self.id_to_token = {i: t for t, i in self.token_to_id.items()}
        self._merge_map = {a + b: (a, b) for a, b in (tuple(m) for m in data["merges"])}
        self._byte_tokens = {str(i) for i in range(256)}
        self.bos_id = self.token_to_id["<s>"]
        self.eos_id = self.token_to_id["</s>"]
        self.pad_id = self.token_to_id["<pad>"]

    def _merge(self, text):
        tokens = [str(b) for b in text.encode("utf-8")]
        while len(tokens) > 1:
            best_rank, best_index = None, -1
            for i in range(len(tokens) - 1):
                rank = self.merge_ranks.get((tokens[i], tokens[i + 1]))
                if rank is not None and (best_rank is None or rank < best_rank):
                    best_rank, best_index = rank, i
            if best_index < 0:
                break
            tokens[best_index:best_index + 2] = [tokens[best_index] + tokens[best_index + 1]]
        return [self.token_to_id.get(t, self.unk_id) for t in tokens]

    def encode_piece(self, piece):
        """Token ids of one piece of text WITHOUT the end-of-word token."""
        ids = self._cache.get(piece)
        if ids is None:
            ids = []
            for k in range(0, len(piece), 200):
                ids.extend(self._merge(piece[k:k + 200]))
            if len(self._cache) < 2_000_000:
                self._cache[piece] = ids
        return ids

    def encode(self, text, max_tokens=None):
        ids = []
        for word in str(text).split():
            ids.extend(self.encode_piece(word))
            ids.append(self.eow_id)
            if max_tokens is not None and len(ids) >= max_tokens:
                break
        return ids if max_tokens is None else ids[:max_tokens]

    def _bytes(self, token):
        if token in self._merge_map:
            a, b = self._merge_map[token]
            return self._bytes(a) + self._bytes(b)
        if token in self._byte_tokens:
            return bytes([int(token)])
        return b""

    def decode(self, ids):
        words, current = [], bytearray()
        for i in ids:
            token = self.id_to_token.get(int(i))
            if token is None or token in SPECIAL_TOKENS:
                continue
            if token == "</w>":
                if current:
                    words.append(current.decode("utf-8", errors="replace"))
                    current = bytearray()
                continue
            current.extend(self._bytes(token))
        if current:
            words.append(current.decode("utf-8", errors="replace"))
        return " ".join(words)

def select_document_tokens(ids, budget, mode):
    """Long documents: mode "head" keeps the first `budget` tokens; "head_tail" keeps the first 60% and the last 40%
    of the budget (judgments state the facts at the start and the decision at the end)."""
    if len(ids) <= budget:
        return list(ids)
    if mode == "head":
        return list(ids[:budget])
    head = int(round(budget * 0.6))
    return list(ids[:head]) + list(ids[len(ids) - (budget - head):])

## 5. Metric: Micro-F1@7, score fusion

In [ ]:
def top_k(scores, k=TOP_K):
    """[queries x pool] scores -> [queries x k] indices of the best k cases, best first (ties broken by index)."""
    idx = np.argpartition(-scores, k, axis=1)[:, :k]
    part = np.take_along_axis(scores, idx, axis=1)
    order = np.lexsort((idx, -part), axis=1)
    return np.take_along_axis(idx, order, axis=1)

def micro_prf(split, ranked):
    tp = n_pred = n_gold = 0
    for q, row in zip(QUERIES[split], ranked):
        gold = set(q["rel_idx"])
        tp += len(gold & set(int(i) for i in row[:TOP_K]))
        n_pred += TOP_K
        n_gold += len(gold)
    p = tp / max(1, n_pred)
    r = tp / max(1, n_gold)
    return {"micro_f1": 2 * p * r / (p + r) if p + r else 0.0, "micro_precision": p, "micro_recall": r}

def standardise(scores):
    """Per query: (score - mean) / std over the pool (excluded entries stay at -inf)."""
    s = scores.astype(np.float64).copy()
    finite = np.isfinite(s)
    s_f = np.where(finite, s, np.nan)
    mu = np.nanmean(s_f, axis=1, keepdims=True)
    sd = np.nanstd(s_f, axis=1, keepdims=True) + 1e-9
    out = (s - mu) / sd
    out[~finite] = -np.inf
    return out

def fuse(neural, lexical, alpha):
    n, l = standardise(neural), standardise(lexical)
    bad = ~np.isfinite(n) | ~np.isfinite(l)
    out = alpha * np.where(bad, 0.0, n) + (1 - alpha) * np.where(bad, 0.0, l)
    out[bad] = -np.inf
    return out

## 6. Phase-1 models

In [ ]:
ARCH_OF_TYPE = {"decoder-only": "A", "decoder-only-gqa": "B", "encoder-decoder": "C", "efficient-local-mqa": "D-alibi"}
ARCH_DESCRIPTION = {"A": "GPT decoder", "B": "GQA/RoPE decoder", "C": "encoder-decoder", "D-alibi": "local MQA/ALiBi decoder"}

def torch_load(path):
    try:
        return torch.load(path, map_location="cpu", weights_only=False)
    except TypeError:
        return torch.load(path, map_location="cpu")

def discover_phase1_checkpoints(root):
    """Finds the models written by the Phase-1 notebooks:
      <...>/<ARCH>/legal-model-final/{model.pt, config.json, tokenizer.json}       (normal case), or
      <...>/<ARCH>/best_model.pt + <...>/common/bpe_tokenizer_<ARCH>/tokenizer.json  (run stopped before the final save)."""
    found = []
    if not os.path.isdir(root):
        return found
    for dirpath, dirnames, filenames in os.walk(root):
        if dirpath[len(root):].count(os.sep) >= 12:
            dirnames[:] = []
        try:
            if os.path.basename(dirpath) == "legal-model-final" and "model.pt" in filenames and "tokenizer.json" in filenames:
                config = load_json_file(os.path.join(dirpath, "config.json"))
                history_path = os.path.join(dirpath, "training_history.json")
                model_file, tokenizer_dir = os.path.join(dirpath, "model.pt"), dirpath
                if os.path.exists(history_path):
                    eval_loss = float(load_json_file(history_path).get("best_eval_loss", "inf"))
                else:
                    eval_loss = float(torch_load(model_file).get("best_eval_loss", "inf"))
            elif "best_model.pt" in filenames and "legal-model-final" not in dirnames:
                common = os.path.join(os.path.dirname(dirpath), "common")
                tokenizer_dirs = [d for d in sorted(glob.glob(os.path.join(common, "bpe_tokenizer_" + os.path.basename(dirpath) + "*")))
                                  if os.path.exists(os.path.join(d, "tokenizer.json"))]
                if not tokenizer_dirs:
                    continue
                saved = torch_load(os.path.join(dirpath, "best_model.pt"))
                config, eval_loss = saved["config"], float(saved.get("eval_loss", "inf"))
                del saved
                model_file, tokenizer_dir = os.path.join(dirpath, "best_model.pt"), tokenizer_dirs[-1]
            else:
                continue
            arch = ARCH_OF_TYPE.get(config.get("type")) or str(config.get("architecture", "?"))[:1]
        except Exception as e:
            print("Skipping", dirpath, "(" + type(e).__name__ + ")")
            continue
        found.append({"arch": arch, "dir": dirpath, "model_file": model_file, "tokenizer_dir": tokenizer_dir, "config": config,
                      "eval_loss": eval_loss, "size_mb": os.path.getsize(model_file) / 1e6})
    return found

PHASE1_FOUND = discover_phase1_checkpoints(INPUT_ROOT)
PHASE1 = {}
print(f"Phase-1 checkpoints found under {INPUT_ROOT}: {len(PHASE1_FOUND)}")
for c in sorted(PHASE1_FOUND, key=lambda c: (c["arch"], c["eval_loss"])):
    print(f"  {ARCH_DESCRIPTION.get(c['arch'], c['arch']):24s} val loss {c['eval_loss']:.4f} | {c['size_mb']:.0f} MB | {c['dir']}")
_chosen = {}
for slot, spec in sorted(PHASE1_MODELS.items(), key=lambda kv: kv[1]["pick"] == "other"):
    arch, choice = spec["architecture"], spec["pick"]
    candidates = [c for c in PHASE1_FOUND if c["arch"] == arch]
    if choice == "other":
        taken = [_chosen[a]["eval_loss"] for a in _chosen if _chosen[a]["arch"] == arch]
        candidates = [c for c in candidates if all(abs(c["eval_loss"] - t) > 1e-6 for t in taken)]
        pick = max(candidates, key=lambda c: c["eval_loss"]) if candidates else None
    else:
        if choice not in ("best", "auto"):
            candidates = [c for c in candidates if choice.lower() in c["dir"].lower()]
        pick = min(candidates, key=lambda c: c["eval_loss"]) if candidates else None
    if pick is not None:
        PHASE1[slot] = _chosen[slot] = dict(pick, slot=slot)
print("Your Phase-1 models:")
for slot in PHASE1_MODELS:
    if slot in PHASE1:
        e = PHASE1[slot]
        print(f"  {slot}: {ARCH_DESCRIPTION.get(e['arch'], e['arch'])}, validation loss {e['eval_loss']:.4f} | {e['dir']}")
    else:
        print(f"  {slot}: NOT FOUND - attach the output of that Phase-1 notebook with 'Add Input'")

TOKENIZERS = {}

def tokenizer_for(slot):
    path = PHASE1[slot]["tokenizer_dir"]
    if path not in TOKENIZERS:
        TOKENIZERS[path] = BPETokenizer(path)
    return TOKENIZERS[path]

class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.eps = eps

    def forward(self, x):
        rms = torch.rsqrt(x.float().pow(2).mean(dim=-1, keepdim=True) + self.eps)
        return (x * rms.to(x.dtype)) * self.weight

class AttentionA(nn.Module):
    def __init__(self, d, n_heads, dropout):
        super().__init__()
        self.n_heads, self.head_dim, self.dropout = n_heads, d // n_heads, dropout
        self.qkv = nn.Linear(d, 3 * d, bias=True)
        self.out = nn.Linear(d, d, bias=True)

    def forward(self, x):
        b, t, c = x.shape
        q, k, v = self.qkv(x).chunk(3, dim=-1)
        q, k, v = [z.view(b, t, self.n_heads, self.head_dim).transpose(1, 2) for z in (q, k, v)]
        y = F.scaled_dot_product_attention(q, k, v, dropout_p=self.dropout if self.training else 0.0, is_causal=True)
        return self.out(y.transpose(1, 2).contiguous().view(b, t, c))

class BlockA(nn.Module):
    def __init__(self, d, n_heads, d_ff, dropout):
        super().__init__()
        self.ln1 = nn.LayerNorm(d)
        self.attn = AttentionA(d, n_heads, dropout)
        self.ln2 = nn.LayerNorm(d)
        self.fc1 = nn.Linear(d, d_ff)
        self.fc2 = nn.Linear(d_ff, d)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x = x + self.dropout(self.attn(self.ln1(x)))
        return x + self.dropout(self.fc2(F.gelu(self.fc1(self.ln2(x)))))

class BackboneA(nn.Module):
    bidirectional = False

    def __init__(self, cfg, vocab_size, n_layers, dropout):
        super().__init__()
        d = cfg["d_model"]
        self.d_model, self.max_tokens = d, cfg["context_length"]
        self.token_embedding = nn.Embedding(vocab_size, d)
        self.position_embedding = nn.Embedding(cfg["context_length"], d)
        self.blocks = nn.ModuleList([BlockA(d, cfg["n_head"], cfg["d_ff"], dropout) for _ in range(n_layers)])
        self.ln_f = nn.LayerNorm(d)

    def layers(self):
        return self.blocks

    def forward(self, input_ids, pad_mask=None):
        positions = torch.arange(input_ids.size(1), device=input_ids.device)
        x = self.token_embedding(input_ids) + self.position_embedding(positions)[None]
        states = []
        for block in self.blocks:
            x = block(x)
            states.append(self.ln_f(x))
        return states

def rotate_half(x):
    return torch.stack((-x[..., 1::2], x[..., ::2]), dim=-1).flatten(-2)

class AttentionB(nn.Module):
    def __init__(self, d, n_heads, n_kv_heads, max_len, dropout):
        super().__init__()
        self.q_heads, self.kv_heads, self.head_dim, self.dropout = n_heads, n_kv_heads, d // n_heads, dropout
        self.q_proj = nn.Linear(d, d, bias=False)
        self.k_proj = nn.Linear(d, n_kv_heads * self.head_dim, bias=False)
        self.v_proj = nn.Linear(d, n_kv_heads * self.head_dim, bias=False)
        self.o_proj = nn.Linear(d, d, bias=False)
        inv_freq = 1.0 / (10000.0 ** (torch.arange(0, self.head_dim, 2).float() / self.head_dim))
        freqs = torch.outer(torch.arange(max_len).float(), inv_freq)
        self.register_buffer("cos", torch.cos(freqs).repeat_interleave(2, dim=-1), persistent=False)
        self.register_buffer("sin", torch.sin(freqs).repeat_interleave(2, dim=-1), persistent=False)

    def forward(self, x):
        b, t, c = x.shape
        q = self.q_proj(x).view(b, t, self.q_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(b, t, self.kv_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(b, t, self.kv_heads, self.head_dim).transpose(1, 2)
        cos, sin = self.cos[:t].to(q.dtype)[None, None], self.sin[:t].to(q.dtype)[None, None]
        q = q * cos + rotate_half(q) * sin
        k = k * cos + rotate_half(k) * sin
        repeat = self.q_heads // self.kv_heads
        if repeat > 1:
            k, v = k.repeat_interleave(repeat, dim=1), v.repeat_interleave(repeat, dim=1)
        y = F.scaled_dot_product_attention(q, k, v, dropout_p=self.dropout if self.training else 0.0, is_causal=True)
        return self.o_proj(y.transpose(1, 2).contiguous().view(b, t, c))

class SwiGLU(nn.Module):
    def __init__(self, d, d_ff):
        super().__init__()
        self.gate = nn.Linear(d, d_ff, bias=False)
        self.up = nn.Linear(d, d_ff, bias=False)
        self.down = nn.Linear(d_ff, d, bias=False)

    def forward(self, x):
        return self.down(F.silu(self.gate(x)) * self.up(x))

class BlockB(nn.Module):
    def __init__(self, d, n_heads, n_kv_heads, d_ff, max_len, dropout):
        super().__init__()
        self.norm1 = RMSNorm(d)
        self.attn = AttentionB(d, n_heads, n_kv_heads, max_len, dropout)
        self.norm2 = RMSNorm(d)
        self.mlp = SwiGLU(d, d_ff)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x = x + self.dropout(self.attn(self.norm1(x)))
        return x + self.dropout(self.mlp(self.norm2(x)))

class BackboneB(nn.Module):
    bidirectional = False

    def __init__(self, cfg, vocab_size, n_layers, dropout):
        super().__init__()
        d = cfg["d_model"]
        self.d_model, self.max_tokens = d, cfg["context_length"]
        self.token_embedding = nn.Embedding(vocab_size, d)
        self.blocks = nn.ModuleList([BlockB(d, cfg["n_head"], cfg["n_kv_head"], cfg["d_ff"], cfg["context_length"], dropout)
                                     for _ in range(n_layers)])
        self.norm_f = RMSNorm(d)

    def layers(self):
        return self.blocks

    def forward(self, input_ids, pad_mask=None):
        x = self.token_embedding(input_ids)
        states = []
        for block in self.blocks:
            x = block(x)
            states.append(self.norm_f(x))
        return states

class AttentionC(nn.Module):
    def __init__(self, d, n_heads, dropout):
        super().__init__()
        self.n_heads, self.head_dim, self.dropout = n_heads, d // n_heads, dropout
        self.q_proj = nn.Linear(d, d)
        self.k_proj = nn.Linear(d, d)
        self.v_proj = nn.Linear(d, d)
        self.o_proj = nn.Linear(d, d)

    def forward(self, x, attn_mask=None):
        b, t, c = x.shape
        q, k, v = [proj(x).view(b, t, self.n_heads, self.head_dim).transpose(1, 2)
                   for proj in (self.q_proj, self.k_proj, self.v_proj)]
        y = F.scaled_dot_product_attention(q, k, v, attn_mask=attn_mask, dropout_p=self.dropout if self.training else 0.0)
        return self.o_proj(y.transpose(1, 2).contiguous().view(b, t, c))

class EncoderBlockC(nn.Module):
    def __init__(self, d, n_heads, d_ff, dropout):
        super().__init__()
        self.self_attn = AttentionC(d, n_heads, dropout)
        self.ln1 = nn.LayerNorm(d)
        self.ff1 = nn.Linear(d, d_ff)
        self.ff2 = nn.Linear(d_ff, d)
        self.ln2 = nn.LayerNorm(d)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, attn_mask=None):
        x = self.ln1(x + self.dropout(self.self_attn(x, attn_mask)))
        return self.ln2(x + self.dropout(self.ff2(F.gelu(self.ff1(x)))))

class BackboneC(nn.Module):
    bidirectional = True

    def __init__(self, cfg, vocab_size, n_layers, dropout):
        super().__init__()
        d = cfg["d_model"]
        self.d_model, self.max_tokens = d, cfg["encoder_length"]
        self.token_embedding = nn.Embedding(vocab_size, d)
        self.encoder = nn.ModuleList([EncoderBlockC(d, cfg["n_head"], cfg["d_ff"], dropout) for _ in range(n_layers)])
        pe = torch.zeros(self.max_tokens, d)
        position = torch.arange(0, self.max_tokens).unsqueeze(1).float()
        div_term = torch.exp(torch.arange(0, d, 2).float() * (-math.log(10000.0) / d))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe, persistent=False)

    def layers(self):
        return self.encoder

    embedding_state = False

    input_scale = 1.0

    def forward(self, input_ids, pad_mask=None):
        x = self.token_embedding(input_ids) * self.input_scale
        states = []
        if self.embedding_state:
            scale = self.token_embedding.weight.detach().float().pow(2).mean().sqrt().clamp_min(1e-12)
            states.append(F.layer_norm(x / scale.to(x.dtype), (self.d_model,)))
        x = x + self.pe[:x.size(1)].to(x.dtype)[None]
        attn_mask = None if pad_mask is None else pad_mask[:, None, None, :]
        for block in self.encoder:
            x = block(x, attn_mask)
            states.append(x)
        return states

class AttentionD(nn.Module):
    def __init__(self, d, n_heads, dropout):
        super().__init__()
        self.n_heads, self.head_dim, self.dropout = n_heads, d // n_heads, dropout
        self.q_proj = nn.Linear(d, d, bias=False)
        self.k_proj = nn.Linear(d, self.head_dim, bias=False)
        self.v_proj = nn.Linear(d, self.head_dim, bias=False)
        self.o_proj = nn.Linear(d, d, bias=False)

    def forward(self, x, attn_bias):
        b, t, c = x.shape
        q = self.q_proj(x).view(b, t, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(b, t, 1, self.head_dim).transpose(1, 2).expand(-1, self.n_heads, -1, -1)
        v = self.v_proj(x).view(b, t, 1, self.head_dim).transpose(1, 2).expand(-1, self.n_heads, -1, -1)
        y = F.scaled_dot_product_attention(q, k, v, attn_mask=attn_bias.to(q.dtype),
                                           dropout_p=self.dropout if self.training else 0.0)
        return self.o_proj(y.transpose(1, 2).contiguous().view(b, t, c))

class BlockD(nn.Module):
    def __init__(self, d, n_heads, d_ff, dropout):
        super().__init__()
        self.norm1 = RMSNorm(d)
        self.attn = AttentionD(d, n_heads, dropout)
        self.norm2 = RMSNorm(d)
        self.fc1 = nn.Linear(d, d_ff)
        self.fc2 = nn.Linear(d_ff, d)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, attn_bias):
        x = x + self.dropout(self.attn(self.norm1(x), attn_bias))
        return x + self.dropout(self.fc2(F.gelu(self.fc1(self.norm2(x)))))

class BackboneD(nn.Module):
    bidirectional = False

    def __init__(self, cfg, vocab_size, n_layers, dropout):
        super().__init__()
        d = cfg["d_model"]
        self.d_model, self.max_tokens, self.window = d, cfg["context_length"], cfg["local_window"]
        self.token_embedding = nn.Embedding(vocab_size, d)
        self.blocks = nn.ModuleList([BlockD(d, cfg["n_head"], cfg["d_ff"], dropout) for _ in range(n_layers)])
        self.norm_f = RMSNorm(d)
        n = cfg["n_head"]
        self.register_buffer("slopes", 2.0 ** (-(8.0 * torch.arange(n) / n)), persistent=False)

    def layers(self):
        return self.blocks

    def attention_bias(self, t, device):
        pos = torch.arange(t, device=device, dtype=torch.float32)
        distance = pos[:, None] - pos[None, :]
        blocked = (distance < 0) | (distance > self.window - 1)
        bias = -self.slopes.to(device)[:, None, None] * distance[None]
        return bias.masked_fill(blocked[None], float("-inf"))[None]

    def forward(self, input_ids, pad_mask=None):
        x = self.token_embedding(input_ids)
        attn_bias = self.attention_bias(input_ids.size(1), input_ids.device)
        states = []
        for block in self.blocks:
            x = block(x, attn_bias)
            states.append(self.norm_f(x))
        return states

BACKBONE_CLASSES = {"A": BackboneA, "B": BackboneB, "C": BackboneC, "D-alibi": BackboneD}

@torch.no_grad()
def encoder_token_check(backbone, length=256):
    """Does the Phase-1 encoder output depend on the input tokens? Two different random token sequences are encoded;
    if the last-layer states at the same positions are almost identical, the encoder only encodes positions."""
    was_training = backbone.training
    backbone.eval()
    g = torch.Generator().manual_seed(0)
    vocab = backbone.token_embedding.weight.shape[0]
    length = min(length, backbone.max_tokens)
    ids = torch.randint(4, vocab, (2, length), generator=g)
    flag, backbone.embedding_state = backbone.embedding_state, False
    last = backbone(ids)[-1].float()
    backbone.embedding_state = flag
    same_position = F.cosine_similarity(last[0], last[1], dim=-1).mean().item()
    other_position = F.cosine_similarity(last[0], last[0].roll(1, 0), dim=-1).mean().item()
    emb_rms = backbone.token_embedding.weight.float().pow(2).mean().sqrt().item()
    pe_rms = backbone.pe[:length].float().pow(2).mean().sqrt().item()
    print(f"Encoder check: token-embedding RMS {emb_rms:.4f} vs position-encoding RMS {pe_rms:.4f} | last layer, "
          f"cosine similarity of two DIFFERENT inputs at the same position {same_position:.3f} "
          f"(neighbouring positions of one input {other_position:.3f}; close to 1.0 = the output ignores the tokens)")
    backbone.train(was_training)

def load_backbone(slot, n_layers=None, dropout=0.1, verbose=True):
    """Build the architecture and load its Phase-1 weights (optionally only the first n layers)."""
    entry = PHASE1[slot]
    arch = entry["arch"]
    cfg = entry["config"]
    state = torch_load(entry["model_file"])["model"]
    total_layers = cfg["encoder_layers"] if arch == "C" else cfg["n_layer"]
    n_layers = total_layers if n_layers is None else min(int(n_layers), total_layers)
    vocab_size = state["token_embedding.weight"].shape[0]
    backbone = BACKBONE_CLASSES[arch](cfg, vocab_size, n_layers, dropout)
    wanted = set(backbone.state_dict().keys())
    missing = wanted - set(state.keys())
    if missing:
        raise RuntimeError(f"Phase-1 checkpoint for {slot} lacks weights: {sorted(missing)[:5]} ...")
    backbone.load_state_dict({k: v for k, v in state.items() if k in wanted}, strict=True)
    if verbose and arch == "C":
        encoder_token_check(backbone)
    if verbose:
        unused = len(state) - len(wanted)
        print(f"Loaded Phase-1 model {slot} ({ARCH_DESCRIPTION.get(arch, arch)}): {n_layers}/{total_layers} layers, "
              f"{sum(p.numel() for p in backbone.parameters()) / 1e6:.2f}M parameters"
              + (f" ({unused} weight tensors not needed, e.g. removed layers or the decoder)" if unused else ""))
    return backbone

class DecoderBlockC(nn.Module):
    def __init__(self, d, n_heads, d_ff, dropout):
        super().__init__()
        self.self_attn = AttentionC(d, n_heads, dropout)
        self.cross_attn = AttentionC(d, n_heads, dropout)
        self.ff1 = nn.Linear(d, d_ff)
        self.ff2 = nn.Linear(d_ff, d)
        self.ln1 = nn.LayerNorm(d)
        self.ln2 = nn.LayerNorm(d)
        self.ln3 = nn.LayerNorm(d)
        self.dropout = nn.Dropout(dropout)

class Seq2SeqC(nn.Module):
    def __init__(self, cfg, vocab_size, dropout):
        super().__init__()
        d = cfg["d_model"]
        self.d_model, self.n_heads = d, cfg["n_head"]
        self.encoder_model = BackboneC(cfg, vocab_size, cfg["encoder_layers"], dropout)
        self.token_embedding = self.encoder_model.token_embedding
        self.decoder = nn.ModuleList([DecoderBlockC(d, cfg["n_head"], cfg["d_ff"], dropout)
                                      for _ in range(cfg.get("decoder_layers", cfg["encoder_layers"]))])
        self.max_src, self.max_tgt = cfg["encoder_length"], cfg.get("decoder_length", cfg["encoder_length"])
        self.register_buffer("tgt_pe", self.encoder_model.pe.new_zeros(0), persistent=False)
        pe = torch.zeros(self.max_tgt, d)
        position = torch.arange(0, self.max_tgt).unsqueeze(1).float()
        div_term = torch.exp(torch.arange(0, d, 2).float() * (-math.log(10000.0) / d))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.tgt_pe = pe

    def layers(self):
        return list(self.encoder_model.encoder) + list(self.decoder)

def load_seq2seq_c(slot, dropout=0.1, verbose=True):
    entry = PHASE1[slot]
    if entry["arch"] != "C":
        raise ValueError("load_seq2seq_c is for the encoder-decoder model C")
    cfg = entry["config"]
    state = torch_load(entry["model_file"])["model"]
    vocab_size = state["token_embedding.weight"].shape[0]
    model = Seq2SeqC(cfg, vocab_size, dropout)
    renamed = {}
    for k, v in state.items():
        if k.startswith("encoder."):
            renamed["encoder_model." + k] = v
        elif k == "token_embedding.weight":
            renamed["encoder_model.token_embedding.weight"] = v
            renamed["token_embedding.weight"] = v
        elif k.startswith("decoder."):
            renamed[k] = v
    wanted = set(model.state_dict().keys())
    missing = wanted - set(renamed)
    if missing:
        raise RuntimeError(f"Phase-1 checkpoint for {slot} lacks weights: {sorted(missing)[:5]} ...")
    model.load_state_dict({k: v for k, v in renamed.items() if k in wanted}, strict=True)
    if verbose:
        encoder_token_check(model.encoder_model)
        print(f"Loaded Phase-1 model {slot} (encoder-decoder): {len(model.encoder_model.encoder)} encoder + "
              f"{len(model.decoder)} decoder layers, {sum(p.numel() for p in model.parameters()) / 1e6:.2f}M parameters")
    return model

## 7. Fine-tuning and adaptation techniques

| Technique | What is trained |
|---|---|
| `full` | every weight |
| `lorma` | low-rank multiplicative update (I + (α/r)·BA)·W0 on q/v (LoRMA+), float32 |
| `ia3` | one learned scale per key, value and feed-forward unit (IA³) |
| `partial` | the top layers only |

In [ ]:
class LoRALinear(nn.Module):
    """LoRA (Hu et al.): additive low-rank update,  W = W0 + (alpha / r) * B A.   B starts at zero.
    With scaling="rs" it is rsLoRA:  W = W0 + (alpha / sqrt(r)) * B A."""

    def __init__(self, base, r, alpha, rows=None, scaling="standard"):
        super().__init__()
        self.base, self.scale = base, (alpha / math.sqrt(r) if scaling == "rs" else alpha / r)
        self.rows = rows or [(0, base.out_features)]
        self.lora_A = nn.ParameterList([nn.Parameter(torch.randn(r, base.in_features) / math.sqrt(base.in_features))
                                        for _ in self.rows])
        self.lora_B = nn.ParameterList([nn.Parameter(torch.zeros(e - s, r)) for s, e in self.rows])

    def forward(self, x):
        y = self.base(x)
        parts, cursor = [], 0
        for (s, e), A, B in zip(self.rows, self.lora_A, self.lora_B):
            if s > cursor:
                parts.append(y[..., cursor:s])
            parts.append(y[..., s:e] + self.scale * F.linear(F.linear(x, A.to(x.dtype)), B.to(x.dtype)))
            cursor = e
        if cursor < y.size(-1):
            parts.append(y[..., cursor:])
        return parts[0] if len(parts) == 1 else torch.cat(parts, dim=-1)

_PERM_INDEX = {}

class LoRMALinear(nn.Module):
    """LoRMA (Low-Rank Multiplicative Adaptation): the pretrained weight is MULTIPLIED by a learned matrix.

      "plus" (LoRMA+):   W = (I + (alpha / r) * B A) W0     B starts at zero, so the multiplier starts as the identity.
      "pi"   (LoRMA-pi): W = P(B A) W0, where P shifts row i of (B A) cyclically by i positions, which makes a
                         low-rank product full rank. B's first column is ones and A[0, 0] = 1 (rest of A zero), so
                         P(B A) starts as the identity.
    Because the multiplier is on the left of W0, it acts on the layer's OUTPUT: h = W0 x, y = M h + bias.
    """

    def __init__(self, base, r, alpha, rows=None, variant="plus"):
        super().__init__()
        self.base, self.scale, self.variant = base, alpha / r, variant
        self.rows = rows or [(0, base.out_features)]
        self.lorma_A, self.lorma_B = nn.ParameterList(), nn.ParameterList()
        for s, e in self.rows:
            m = e - s
            if variant == "plus":
                A, B = torch.randn(r, m) / math.sqrt(m), torch.zeros(m, r)
            elif variant == "pi":
                A, B = torch.zeros(r, m), torch.randn(m, r) * 0.02
                A[0, 0] = 1.0
                B[:, 0] = 1.0
            else:
                raise ValueError("lorma_variant must be 'plus' or 'pi'")
            self.lorma_A.append(nn.Parameter(A))
            self.lorma_B.append(nn.Parameter(B))

    @staticmethod
    def _permuted(BA):
        m = BA.size(0)
        key = (m, BA.device)
        if key not in _PERM_INDEX:
            i = torch.arange(m, device=BA.device)
            _PERM_INDEX[key] = (i[None, :] - i[:, None]) % m
        return torch.gather(BA, 1, _PERM_INDEX[key])

    def forward(self, x):
        h = F.linear(x, self.base.weight)
        parts, cursor = [], 0
        for (s, e), A, B in zip(self.rows, self.lorma_A, self.lorma_B):
            if s > cursor:
                parts.append(h[..., cursor:s])
            hs = h[..., s:e]
            if self.variant == "plus":
                upd = self.scale * F.linear(F.linear(hs.float(), A.float()), B.float())
                parts.append((hs.float() + upd).to(hs.dtype))
            else:
                parts.append(F.linear(hs, self._permuted(B @ A).to(hs.dtype)))
            cursor = e
        if cursor < h.size(-1):
            parts.append(h[..., cursor:])
        y = parts[0] if len(parts) == 1 else torch.cat(parts, dim=-1)
        return y if self.base.bias is None else y + self.base.bias

class AdapterLinear(nn.Module):
    """Houlsby-style bottleneck adapter placed on a sub-layer's output, before the residual connection:
    y = h + up(GELU(down(h))),  h = base(x).   `up` starts at zero, so the adapter starts as the identity."""

    def __init__(self, base, bottleneck):
        super().__init__()
        self.base = base
        self.adapter_down = nn.Linear(base.out_features, bottleneck)
        self.adapter_up = nn.Linear(bottleneck, base.out_features)
        nn.init.normal_(self.adapter_down.weight, std=0.02)
        nn.init.zeros_(self.adapter_down.bias)
        nn.init.zeros_(self.adapter_up.weight)
        nn.init.zeros_(self.adapter_up.bias)

    def forward(self, x):
        h = self.base(x)
        return h + self.adapter_up(F.gelu(self.adapter_down(h)))

class IA3Linear(nn.Module):
    """IA3 (Liu et al., "(IA)^3"): learned vectors that RESCALE activations, initialised to ones (= no change).
    mode "out": rescale the layer's output (used for attention keys and values; `rows` limits it to part of a fused
    matrix).  mode "in": rescale the layer's input (used for the feed-forward hidden activations, before fc2)."""

    def __init__(self, base, mode, rows=None):
        super().__init__()
        self.base, self.mode = base, mode
        if mode == "in":
            self.ia3_scale = nn.Parameter(torch.ones(base.in_features))
            self.rows = None
        else:
            self.rows = rows or [(0, base.out_features)]
            index = torch.cat([torch.arange(s, e) for s, e in self.rows])
            self.register_buffer("row_index", index, persistent=False)
            self.ia3_scale = nn.Parameter(torch.ones(len(index)))

    def forward(self, x):
        if self.mode == "in":
            return self.base(x * self.ia3_scale.to(x.dtype))
        y = self.base(x)
        full = torch.ones(y.size(-1), device=y.device, dtype=y.dtype).scatter(0, self.row_index, self.ia3_scale.to(y.dtype))
        return y * full

QV_TARGETS = {"A": ["attn.qkv"], "B": ["attn.q_proj", "attn.v_proj"], "C": ["self_attn.q_proj", "self_attn.v_proj"],
              "D-alibi": ["attn.q_proj", "attn.v_proj"]}
ALL_TARGETS = {"A": ["attn.qkv", "attn.out", "fc1", "fc2"],
               "B": ["attn.q_proj", "attn.k_proj", "attn.v_proj", "attn.o_proj", "mlp.gate", "mlp.up", "mlp.down"],
               "C": ["self_attn.q_proj", "self_attn.k_proj", "self_attn.v_proj", "self_attn.o_proj", "ff1", "ff2",
                     "cross_attn.q_proj", "cross_attn.k_proj", "cross_attn.v_proj", "cross_attn.o_proj"],
               "D-alibi": ["attn.q_proj", "attn.k_proj", "attn.v_proj", "attn.o_proj", "fc1", "fc2"]}
SUBLAYER_OUTPUTS = {"A": ["attn.out", "fc2"], "B": ["attn.o_proj", "mlp.down"], "C": ["self_attn.o_proj", "cross_attn.o_proj", "ff2"],
                    "D-alibi": ["attn.o_proj", "fc2"]}
IA3_TARGETS = {"A": [("attn.qkv", "out"), ("fc2", "in")], "B": [("attn.k_proj", "out"), ("attn.v_proj", "out"), ("mlp.down", "in")],
               "C": [("self_attn.k_proj", "out"), ("self_attn.v_proj", "out"), ("ff2", "in")],
               "D-alibi": [("attn.k_proj", "out"), ("attn.v_proj", "out"), ("fc2", "in")]}
NEW_WEIGHT_MARKERS = ("lora_", "lorma_", "adapter_", "ia3_")

FFN_OUTPUTS = {"A": ["fc2"], "B": ["mlp.down"], "C": ["ff2"], "D-alibi": ["fc2"]}

def _has(module, path):
    for part in path.split("."):
        if not hasattr(module, part):
            return False
        module = getattr(module, part)
    return True

def _get(module, path):
    for part in path.split("."):
        module = getattr(module, part)
    return module

def _set(module, path, value):
    parts = path.split(".")
    for part in parts[:-1]:
        module = getattr(module, part)
    setattr(module, parts[-1], value)

def apply_technique(backbone, arch, cfg):
    """Freeze the pretrained model, then make trainable exactly what the chosen technique allows."""
    technique = cfg["technique"]
    for p in backbone.parameters():
        p.requires_grad = False
    blocks = backbone.layers()
    norm_types = (nn.LayerNorm, RMSNorm)
    if technique == "full":
        for p in backbone.parameters():
            p.requires_grad = True
    elif technique == "partial":
        for block in list(blocks)[-int(cfg["partial_top_layers"]):]:
            for p in block.parameters():
                p.requires_grad = True
        for name in ("ln_f", "norm_f"):
            if hasattr(backbone, name):
                for p in getattr(backbone, name).parameters():
                    p.requires_grad = True
    elif technique == "bitfit":
        for name, p in backbone.named_parameters():
            if name.endswith(".bias"):
                p.requires_grad = True
        for module in backbone.modules():
            if isinstance(module, norm_types):
                for p in module.parameters():
                    p.requires_grad = True
    elif technique in ("lora", "lorma"):
        targets = (QV_TARGETS if cfg["peft_targets"] == "qv" else ALL_TARGETS)[arch]
        for block in blocks:
            for path in targets:
                if not _has(block, path):
                    continue
                base = _get(block, path)
                rows = None
                if arch == "A" and path == "attn.qkv" and cfg["peft_targets"] == "qv":
                    d = base.in_features
                    rows = [(0, d), (2 * d, 3 * d)]
                if technique == "lora":
                    wrapped = LoRALinear(base, cfg["lora_r"], cfg["lora_alpha"], rows, cfg.get("lora_scaling", "standard"))
                else:
                    wrapped = LoRMALinear(base, cfg["lora_r"], cfg["lora_alpha"], rows, cfg["lorma_variant"])
                _set(block, path, wrapped)
    elif technique == "ia3":
        for block in blocks:
            for path, mode in IA3_TARGETS[arch]:
                base = _get(block, path)
                rows = None
                if arch == "A" and path == "attn.qkv":
                    d = base.in_features
                    rows = [(d, 2 * d), (2 * d, 3 * d)]
                _set(block, path, IA3Linear(base, mode, rows))
    elif technique in ("adapter", "pfeiffer"):
        for block in blocks:
            for path in (SUBLAYER_OUTPUTS if technique == "adapter" else FFN_OUTPUTS)[arch]:
                if _has(block, path):
                    _set(block, path, AdapterLinear(_get(block, path), cfg["adapter_dim"]))
        for module in backbone.modules():
            if isinstance(module, norm_types):
                for p in module.parameters():
                    p.requires_grad = True
    elif technique != "frozen":
        raise ValueError("Unknown technique: " + str(technique))
    for name, p in backbone.named_parameters():
        if any(marker in name for marker in NEW_WEIGHT_MARKERS):
            p.requires_grad = True
    return backbone

## 8. Retrieval encoder

In [ ]:
class RetrievalEncoder(nn.Module):
    """Phase-1 language model -> one vector per window of a case (a bi-encoder: queries and candidates are encoded
    by the same model, independently, so all candidate vectors can be computed once and compared by cosine).

    Window vector: learned weighted average of ALL layers ("scalar mix"), averaged over the window's tokens,
    LayerNorm, L2-normalised. Case vector: mean of its window vectors, L2-normalised.
    """

    def __init__(self, cfg, verbose=True):
        super().__init__()
        self.slot = cfg["backbone"]
        self.arch = PHASE1[self.slot]["arch"]
        self.backbone = load_backbone(self.slot, cfg["n_layers"], cfg["dropout"], verbose=verbose)
        if self.arch == "C":
            scale = cfg.get("c_input_scale", 1.0)
            if scale == "match":
                emb_rms = self.backbone.token_embedding.weight.detach().float().pow(2).mean().sqrt().item()
                pe_rms = self.backbone.pe.float().pow(2).mean().sqrt().item()
                scale = max(1.0, pe_rms / max(emb_rms, 1e-8))
            self.backbone.input_scale = float(scale)
            if cfg.get("embedding_state"):
                self.backbone.embedding_state = True
            if verbose and float(scale) != 1.0:
                print(f"Model C: encoder input embeddings scaled by {float(scale):.1f}")
        elif cfg.get("embedding_state"):
            raise ValueError("embedding_state is only implemented for model C")
        if self.backbone.max_tokens < CHUNK_TOKENS:
            raise ValueError(f"CHUNK_TOKENS {CHUNK_TOKENS} > the model's limit {self.backbone.max_tokens}")
        apply_technique(self.backbone, self.arch, cfg)
        n_states = len(self.backbone.layers()) + int(bool(getattr(self.backbone, "embedding_state", False)))
        self.mix = nn.Parameter(torch.zeros(n_states))
        self.state_norm = bool(cfg.get("pooled_state_norm", False))
        if self.state_norm and getattr(self.backbone, "embedding_state", False):
            with torch.no_grad():
                self.mix[0] = math.log(max(1, len(self.backbone.layers())))
        self.norm = nn.LayerNorm(self.backbone.d_model)

    def chunk_vectors(self, ids, mask):
        """[N, T] token ids of N windows -> [N, d] unit vectors (float32)."""
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
            states = self.backbone(ids, mask)
        w = torch.softmax(self.mix, dim=0)
        assert len(states) == len(w), (len(states), len(w))
        m = mask[:, :, None].float()
        if self.state_norm:
            pooled = [F.layer_norm((s.float() * m).sum(1) / m.sum(1).clamp(min=1.0), (s.size(-1),)) for s in states]
            v = sum(a * p for a, p in zip(w, pooled))
        else:
            h = sum(a * s.float() for a, s in zip(w, states))
            v = (h * m).sum(1) / m.sum(1).clamp(min=1.0)
        return F.normalize(self.norm(v), dim=-1)

    def doc_vectors(self, ids, mask, owner, n_docs):
        """Windows of several cases -> [n_docs, d] unit case vectors (mean of each case's window vectors)."""
        c = self.chunk_vectors(ids, mask)
        s = torch.zeros(n_docs, c.size(1), device=c.device).index_add_(0, owner, c)
        return F.normalize(s, dim=-1)

## 9. Embedding, similarity and contrastive training

In [ ]:
TOKENS = {}

def tokenized(slot):
    """{split: {"q": [[np.int32 window ids]], "c": [[...]]}} - up to MAX_CHUNKS windows per case, CHUNK_TOKENS each."""
    tokenizer_dir = PHASE1[slot]["tokenizer_dir"]
    sizes = "-".join(f"{len(QUERIES[s])}.{len(POOLS[s]['ids'])}" for s in ("train", "dev", "test"))
    key = hashlib.md5(f"{tokenizer_dir}|{sizes}|{WINDOW_WORDS}|{CHUNK_TOKENS}|{MAX_CHUNKS}".encode()).hexdigest()[:12]
    if key in TOKENS:
        return TOKENS[key]
    path = os.path.join(CACHE_DIR, f"tokens_{key}.pkl")
    if os.path.exists(path):
        with open(path, "rb") as f:
            TOKENS[key] = pickle.load(f)
        print(f"tokenised cases for model {slot}: loaded from {path}")
        return TOKENS[key]
    tok = tokenizer_for(slot)
    enc = lambda text: [np.asarray(tok.encode(w, CHUNK_TOKENS) or [tok.eow_id], dtype=np.int32) for w in windows(text, MAX_CHUNKS)]
    out = {}
    for split in ("train", "dev", "test"):
        out[split] = {"q": [enc(q["text"]) for q in tqdm(QUERIES[split], desc=f"tokenising {split} queries for {slot}")],
                      "c": [enc(t) for t in tqdm(POOLS[split]["texts"], desc=f"tokenising {split} candidates for {slot}")]}
    with open(path + ".tmp", "wb") as f:
        pickle.dump(out, f, protocol=4)
    os.replace(path + ".tmp", path)
    TOKENS[key] = out
    return out

def spread(chunks, k):
    """k windows spread evenly over the stored windows of a case."""
    if len(chunks) <= k:
        return list(chunks)
    return [chunks[i] for i in np.unique(np.round(np.linspace(0, len(chunks) - 1, k)).astype(int))]

def chunk_batch(seqs):
    width = max(len(s) for s in seqs)
    ids = np.full((len(seqs), width), PAD_ID, dtype=np.int64)
    mask = np.zeros((len(seqs), width), dtype=bool)
    for i, s in enumerate(seqs):
        ids[i, :len(s)], mask[i, :len(s)] = s, True
    return torch.from_numpy(ids).to(DEVICE), torch.from_numpy(mask).to(DEVICE)

@torch.no_grad()
def embed(model, cases, k, batch_chunks=64, desc="embedding"):
    """cases: list of window lists. -> (case vectors [N, d] float32, window vectors [M, d] float16, offsets [N + 1])."""
    model.eval()
    chosen = [spread(c, k) for c in cases]
    flat = [(i, w) for i, c in enumerate(chosen) for w in c]
    order = sorted(range(len(flat)), key=lambda j: len(flat[j][1]))
    vecs = None
    for s in tqdm(range(0, len(order), batch_chunks), desc=desc, leave=False):
        idx = order[s:s + batch_chunks]
        ids, mask = chunk_batch([flat[j][1] for j in idx])
        v = model.chunk_vectors(ids, mask).float().cpu().numpy()
        if vecs is None:
            vecs = np.zeros((len(flat), v.shape[1]), dtype=np.float32)
        vecs[idx] = v
    owner = np.asarray([i for i, _ in flat])
    docs = np.zeros((len(cases), vecs.shape[1]), dtype=np.float32)
    np.add.at(docs, owner, vecs)
    docs /= np.linalg.norm(docs, axis=1, keepdims=True).clip(min=1e-9)
    offsets = np.concatenate([[0], np.cumsum([len(c) for c in chosen])])
    return docs, vecs.astype(np.float16), offsets

def neural_scores(q, c, kind):
    """q, c: outputs of embed(). kind "mean": cosine of case vectors; "maxsim": for every query window the best
    matching candidate window, averaged over the query's windows."""
    if kind == "mean":
        return q[0] @ c[0].T
    qv, qo = q[1].astype(np.float32), q[2]
    cv, co = c[1].astype(np.float32), c[2]
    out = np.zeros((len(qo) - 1, len(co) - 1), dtype=np.float32)
    for i in range(len(qo) - 1):
        s = qv[qo[i]:qo[i + 1]] @ cv.T
        out[i] = np.maximum.reduceat(s, co[:-1], axis=1).mean(0)
    return out

def exp_dir(name):
    path = os.path.join(WORK_ROOT, name)
    os.makedirs(path, exist_ok=True)
    return path

def build_model(cfg, verbose=True):
    return RetrievalEncoder(cfg, verbose=verbose).to(DEVICE)

def count_parameters(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return {"total": total, "trainable": trainable, "trainable_percent": 100.0 * trainable / max(1, total)}

def trainable_state(model):
    return {name: p.detach().cpu().clone() for name, p in model.named_parameters() if p.requires_grad}

def load_trainable(model, state):
    result = model.load_state_dict(state, strict=False)
    if result.unexpected_keys:
        raise RuntimeError("Saved weights do not fit this model: " + str(result.unexpected_keys[:5]))

def save_atomic(obj, path):
    torch.save(obj, path + ".tmp")
    os.replace(path + ".tmp", path)

def config_signature(cfg, extra=""):
    entry = PHASE1[cfg["backbone"]]
    payload = {"cfg": {k: v for k, v in cfg.items() if k != "name" and not (k == "init_from" and v is None)}, "phase1": [entry["config"], entry["eval_loss"]],
               "data": {s: [len(QUERIES[s]), len(POOLS[s]["ids"])] for s in QUERIES},
               "windows": [WINDOW_WORDS, CHUNK_TOKENS, MAX_CHUNKS, EVAL_CHUNKS, TRAIN_CHUNKS], "extra": extra}
    return hashlib.md5(json.dumps(payload, sort_keys=True, default=str).encode()).hexdigest()

def optimizer_groups(model, cfg):
    peft_lr = cfg["lr_peft"] if cfg["technique"] in ("bitfit", "adapter", "ia3") else cfg["lr_backbone"]
    buckets = defaultdict(list)
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if not name.startswith("backbone."):
            lr = cfg["lr_head"]
        elif any(m in name for m in NEW_WEIGHT_MARKERS):
            lr = cfg["lr_peft"]
        else:
            lr = peft_lr
        decay = cfg["weight_decay"] if p.ndim >= 2 else 0.0
        buckets[(lr, decay)].append(p)
    return [{"params": params, "lr": lr, "base_lr": lr, "weight_decay": decay} for (lr, decay), params in buckets.items()]

def lr_factor(step, total_steps, warmup_frac):
    warmup = max(1, int(total_steps * warmup_frac))
    if step < warmup:
        return (step + 1) / warmup
    progress = min(1.0, (step - warmup) / max(1, total_steps - warmup))
    return 0.05 + 0.95 * 0.5 * (1.0 + math.cos(math.pi * progress))

def release(*objects):
    for obj in objects:
        del obj
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

def dev_check(model, slot, k=EVAL_CHUNKS):
    """Quick dev score of the model alone: Micro-F1@7 of the case-vector cosine (k windows per case)."""
    t = tokenized(slot)["dev"]
    q = embed(model, t["q"], k, desc="dev queries")
    c = embed(model, t["c"], k, desc="dev candidates")
    return micro_prf("dev", top_k(exclude_self("dev", neural_scores(q, c, "mean"))))

def hard_negatives(split, n):
    """For every query: the BM25 top-n candidates that are NOT relevant (and not the query itself)."""
    ranked = top_k(BM25[split], min(n + 50, BM25[split].shape[1] - 1))
    out = []
    for qi, (q, row) in enumerate(zip(QUERIES[split], ranked)):
        gold = set(q["rel_idx"])
        out.append([int(j) for j in row if int(j) not in gold and np.isfinite(BM25[split][qi, j])][:n])
    return out

def train_model(cfg, signature, teacher_vectors=None):
    """In-batch contrastive training (InfoNCE): every query must rank its relevant case above the relevant cases of
    the other queries in the batch and above one BM25 hard negative per query. Other relevant cases of the same query
    are masked (they are not negatives). With a teacher, the student also matches the teacher's similarity
    distribution over the same batch (knowledge distillation). One dev check per epoch; the best epoch is kept."""
    folder = exp_dir(cfg["name"])
    last_path, model_path = os.path.join(folder, "last.pt"), os.path.join(folder, "model.pt")
    torch.manual_seed(cfg["seed"])
    model = build_model(cfg)
    if cfg.get("init_from"):
        start = os.path.join(WORK_ROOT, cfg["init_from"], "model.pt")
        load_trainable(model, torch_load(start)["model"])
        print(f"[{cfg['name']}] continuing from the trained {cfg['init_from']} model")
    params = count_parameters(model)
    print(f"[{cfg['name']}] parameters: {params['total'] / 1e6:.2f}M total, {params['trainable'] / 1e6:.3f}M trained "
          f"({params['trainable_percent']:.2f}%)")
    tok = tokenized(cfg["backbone"])["train"]
    pairs = [(qi, pi) for qi, q in enumerate(QUERIES["train"]) for pi in q["rel_idx"]]
    if not pairs:
        raise RuntimeError("no training pairs (no relevant candidates found in the train pool)")
    negs = hard_negatives("train", cfg["hard_negative_pool"])
    golds = [set(q["rel_idx"]) for q in QUERIES["train"]]
    B = cfg["queries_per_batch"]
    epoch_pairs = []
    for e in range(cfg["epochs"]):
        rng = np.random.RandomState(cfg["seed"] * 1000 + e)
        order = rng.permutation(len(pairs))[:cfg["max_pairs_per_epoch"]]
        epoch_pairs.append([pairs[i] for i in order])
    steps_per_epoch = [math.ceil(len(p) / B) for p in epoch_pairs]
    total_steps = sum(steps_per_epoch)
    print(f"[{cfg['name']}] {len(pairs):,} (query, relevant case) pairs; {len(epoch_pairs[0]):,} per epoch, "
          f"{steps_per_epoch[0]:,} steps of {B} queries")
    optimizer = torch.optim.AdamW(optimizer_groups(model, cfg), betas=(0.9, 0.98))
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
    state = {"epoch": 0, "step": 0, "history": [], "best_f1": -1.0, "best_epoch": 0, "bad_epochs": 0,
             "seconds": 0.0, "peak_memory_mb": 0.0, "skipped": 0}
    if os.path.exists(last_path):
        saved = torch_load(last_path)
        if saved.get("signature") == signature:
            load_trainable(model, saved["model"])
            optimizer.load_state_dict(saved["optimizer"])
            scaler.load_state_dict(saved["scaler"])
            state = saved["state"]
            print(f"[{cfg['name']}] resuming after epoch {state['epoch']}")
        del saved
    if DEVICE.type == "cuda":
        torch.cuda.reset_peak_memory_stats()
    if state["epoch"] == 0 and not state["history"]:
        before = dev_check(model, cfg["backbone"])
        state["history"].append({"epoch": 0, "dev_micro_f1": before["micro_f1"]})
        if cfg.get("init_from"):
            state.update(best_f1=before["micro_f1"], best_epoch=0)
            save_atomic({"model": trainable_state(model), "signature": signature, "epoch": 0}, model_path)
        print(f"[{cfg['name']}] before training ({'the ' + cfg['init_from'] + ' model' if cfg.get('init_from') else 'Phase-1 model as it is'}): "
              f"dev Micro-F1@7 {before['micro_f1']:.4f}", flush=True)

    for epoch in range(state["epoch"], cfg["epochs"]):
        if state["bad_epochs"] >= cfg["patience"]:
            print(f"[{cfg['name']}] early stop: no improvement for {cfg['patience']} epochs.")
            break
        model.train()
        rng = np.random.RandomState(cfg["seed"] * 7919 + epoch)
        torch.manual_seed(cfg["seed"] * 1000 + epoch)
        t0, loss_sum, n_loss = time.time(), 0.0, 0
        batches = [epoch_pairs[epoch][i:i + B] for i in range(0, len(epoch_pairs[epoch]), B)]
        bar = tqdm(batches, desc=f"{cfg['name']} epoch {epoch + 1}/{cfg['epochs']}", leave=False)
        for batch in bar:
            for g in optimizer.param_groups:
                g["lr"] = g["base_lr"] * lr_factor(state["step"], total_steps, cfg["warmup_frac"])
            qs = [qi for qi, _ in batch]
            docs = [pi for _, pi in batch] + [negs[qi][rng.randint(len(negs[qi]))] if negs[qi] else int(rng.randint(len(POOLS["train"]["ids"])))
                                             for qi in qs]
            pick = lambda chunks: [chunks[j] for j in sorted(rng.choice(len(chunks), min(TRAIN_CHUNKS, len(chunks)), replace=False))]
            seqs, owner = [], []
            for n, ch in enumerate([tok["q"][qi] for qi in qs] + [tok["c"][di] for di in docs]):
                for w in pick(ch):
                    seqs.append(w)
                    owner.append(n)
            ids, mask = chunk_batch(seqs)
            vec = model.doc_vectors(ids, mask, torch.tensor(owner, device=DEVICE), len(qs) + len(docs))
            qv, dv = vec[:len(qs)], vec[len(qs):]
            blocked = torch.tensor([[(d in golds[qi]) and j != i for j, d in enumerate(docs)] for i, qi in enumerate(qs)],
                                   device=DEVICE)
            logits = (qv @ dv.T / cfg["temperature"]).masked_fill(blocked, -1e4)
            target = torch.arange(len(qs), device=DEVICE)
            loss = F.cross_entropy(logits, target)
            if teacher_vectors is not None:
                tq = torch.from_numpy(teacher_vectors["q"][qs]).to(DEVICE)
                td = torch.from_numpy(teacher_vectors["c"][docs]).to(DEVICE)
                t_logits = (tq @ td.T / cfg["kd_temperature"]).masked_fill(blocked, -1e4)
                kd = F.kl_div(F.log_softmax(logits, dim=-1), F.softmax(t_logits, dim=-1), reduction="batchmean")
                loss = (1 - cfg["kd_weight"]) * loss + cfg["kd_weight"] * kd
            optimizer.zero_grad(set_to_none=True)
            state["step"] += 1
            if not torch.isfinite(loss):
                state["skipped"] += 1
                if state["skipped"] in (1, 10, 100):
                    print(f"[{cfg['name']}] WARNING: non-finite loss, batch skipped ({state['skipped']} so far)", flush=True)
                continue
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], cfg["grad_clip"])
            scaler.step(optimizer)
            scaler.update()
            loss_sum += loss.item()
            n_loss += 1
            if n_loss % 10 == 0:
                bar.set_postfix(loss=f"{loss_sum / n_loss:.4f}")
        bar.close()
        seconds = time.time() - t0
        dev = dev_check(model, cfg["backbone"])
        record = {"epoch": epoch + 1, "train_loss": loss_sum / max(1, n_loss), "dev_micro_f1": dev["micro_f1"],
                  "dev_precision": dev["micro_precision"], "dev_recall": dev["micro_recall"], "seconds": seconds}
        message = (f"[{cfg['name']}] epoch {epoch + 1}/{cfg['epochs']} | train loss {record['train_loss']:.4f} | dev Micro-F1@7 "
                   f"{dev['micro_f1']:.4f} (model alone, {EVAL_CHUNKS} windows per case)")
        if dev["micro_f1"] > state["best_f1"]:
            state.update(best_f1=dev["micro_f1"], best_epoch=epoch + 1, bad_epochs=0)
            save_atomic({"model": trainable_state(model), "signature": signature, "epoch": epoch + 1}, model_path)
            message += "  <- best"
        else:
            state["bad_epochs"] += 1
        state["history"].append(record)
        state["epoch"] = epoch + 1
        state["seconds"] += seconds
        if DEVICE.type == "cuda":
            state["peak_memory_mb"] = max(state["peak_memory_mb"], torch.cuda.max_memory_allocated() / 1e6)
        save_atomic({"model": trainable_state(model), "optimizer": optimizer.state_dict(), "scaler": scaler.state_dict(),
                     "state": state, "signature": signature}, last_path)
        print(message + f" | {seconds / 60:.1f} min", flush=True)
    load_trainable(model, torch_load(model_path)["model"])
    if os.path.exists(last_path):
        os.remove(last_path)
    info = {"best_epoch": state["best_epoch"], "best_dev_f1_quick": state["best_f1"], "history": state["history"],
            "train_seconds": state["seconds"], "peak_memory_mb": state["peak_memory_mb"], "parameters": params,
            "train_pairs": len(pairs), "skipped_batches": state["skipped"]}
    return model, info

def load_model(cfg):
    model = build_model(cfg, verbose=False)
    load_trainable(model, torch_load(os.path.join(exp_dir(cfg["name"]), "model.pt"))["model"])
    return model

## 10. Run the experiments

For each experiment: train (best epoch on dev) → embed dev and test → choose similarity type and BM25 weight on dev → test top-7 → zip. Finished experiments are loaded from disk.

In [ ]:
RESULTS = {}
BM25_DEV = micro_prf("dev", top_k(BM25["dev"]))
print(f"BM25 alone (lexical baseline, no model): dev Micro-F1@7 {BM25_DEV['micro_f1']:.4f} "
      f"(P {BM25_DEV['micro_precision']:.4f}, R {BM25_DEV['micro_recall']:.4f})")

def write_predictions(cfg, folder, ranked, dev_f1):
    """prediction.json = {query id: [7 candidate ids, best first]} for every test query: exactly 7 unique ids, all
    from the test candidate pool, never the query itself. Zipped as <short name>.zip (at most 64 characters)."""
    ids = POOLS["test"]["ids"]
    output = {}
    for q, row in zip(QUERIES["test"], ranked):
        output[q["id"]] = [ids[int(j)] for j in row[:TOP_K]]
    pool = set(ids)
    assert set(output) == {q["id"] for q in QUERIES["test"]}
    for qid, lst in output.items():
        assert len(lst) == TOP_K and len(set(lst)) == TOP_K and all(c in pool for c in lst) and qid not in lst, qid
    prediction_path = os.path.join(folder, "prediction.json")
    with open(prediction_path, "w", encoding="utf-8") as f:
        json.dump(output, f, ensure_ascii=False)
    zip_name = submission_name(cfg, dev_f1)
    for old in glob.glob(os.path.join(folder, "p7_*.zip")) + glob.glob(os.path.join(SUBMISSION_DIR, f"p7_{cfg['name'].split('_')[0]}_*.zip")):
        os.remove(old)
    zip_path = os.path.join(folder, zip_name)
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
        z.write(prediction_path, arcname="prediction.json")
    with zipfile.ZipFile(zip_path, "r") as z:
        assert z.namelist() == ["prediction.json"]
        assert json.loads(z.read("prediction.json").decode("utf-8")) == output
    shutil.copy2(zip_path, os.path.join(SUBMISSION_DIR, zip_name))
    return os.path.join(SUBMISSION_DIR, zip_name)

def cached_embeddings(model, cfg, signature, split, kind, k):
    path = os.path.join(exp_dir(cfg["name"]), f"emb_{split}_{kind}_{k}.npz")
    if os.path.exists(path):
        z = np.load(path, allow_pickle=False)
        if str(z["signature"]) == signature:
            return z["docs"], z["chunks"], z["offsets"]
    t = tokenized(cfg["backbone"])[split][kind]
    docs, chunks, offsets = embed(model, t, k, desc=f"{cfg['name']} {split} {'queries' if kind == 'q' else 'candidates'}")
    np.savez(path, docs=docs, chunks=chunks, offsets=offsets, signature=np.asarray(signature))
    return docs, chunks, offsets

def teacher_vectors_for(cfg):
    """The teacher's case vectors for all training queries and candidates (MAX_CHUNKS windows), for distillation."""
    teacher_cfg = experiment_config(cfg["teacher"])
    teacher_sig = RESULTS[cfg["teacher"]]["signature"]
    teacher = load_model(teacher_cfg)
    q = cached_embeddings(teacher, teacher_cfg, teacher_sig, "train", "q", MAX_CHUNKS)[0]
    c = cached_embeddings(teacher, teacher_cfg, teacher_sig, "train", "c", MAX_CHUNKS)[0]
    print(f"[{cfg['name']}] distilling from {cfg['teacher']} ({count_parameters(teacher)['total'] / 1e6:.1f}M parameters)")
    release(teacher)
    return {"q": q.astype(np.float32), "c": c.astype(np.float32)}

def run_experiment(name):
    if name in RESULTS:
        return RESULTS[name]
    cfg = experiment_config(name)
    if cfg["backbone"] not in PHASE1:
        print(f"[{name}] SKIPPED: no Phase-1 checkpoint for model {cfg['backbone']} is attached.")
        return None
    teacher_sig = ""
    if cfg["teacher"]:
        dep = run_experiment(cfg["teacher"])
        if dep is None:
            print(f"[{name}] SKIPPED: its teacher {cfg['teacher']} could not be run.")
            return None
        teacher_sig = dep["signature"]
    if cfg.get("init_from"):
        base = run_experiment(cfg["init_from"])
        if base is None:
            print(f"[{name}] SKIPPED: {cfg['init_from']} could not be run.")
            return None
        teacher_sig += "|init:" + base["signature"]
    folder = exp_dir(name)
    signature = config_signature(cfg, teacher_sig)
    results_path = os.path.join(folder, "results.json")
    if os.path.exists(results_path):
        saved = load_json_file(results_path)
        if saved.get("signature") == signature and os.path.exists(saved.get("submission", "")):
            print(f"[{name}] already finished: dev Micro-F1@7 {saved['dev']['micro_f1']:.4f} (loaded from disk)")
            RESULTS[name] = saved
            return saved

    model_path, info_path = os.path.join(folder, "model.pt"), os.path.join(folder, "train_info.json")
    trained = (os.path.exists(model_path) and os.path.exists(info_path) and not os.path.exists(os.path.join(folder, "last.pt"))
               and torch_load(model_path).get("signature") == signature)
    if trained:
        model = load_model(cfg)
        info = load_json_file(info_path)
        print(f"[{name}] model already trained; loading it.")
    else:
        teacher = teacher_vectors_for(cfg) if cfg["teacher"] else None
        model, info = train_model(cfg, signature, teacher)
        write_json(info, info_path)

    emb = {(s, k): cached_embeddings(model, cfg, signature, s, k, MAX_CHUNKS) for s in ("dev", "test") for k in ("q", "c")}
    t0 = time.time()
    neural = {(s, kind): exclude_self(s, neural_scores(emb[(s, "q")], emb[(s, "c")], kind))
              for s in ("dev", "test") for kind in ("mean", "maxsim")}
    grid = {}
    for kind in ("mean", "maxsim"):
        grid[f"{kind}|model_only"] = micro_prf("dev", top_k(neural[("dev", kind)]))
        for a in ALPHA_GRID:
            grid[f"{kind}|{a}"] = micro_prf("dev", top_k(fuse(neural[("dev", kind)], BM25["dev"], a)))
    fused = {k: v for k, v in grid.items() if not k.endswith("model_only")}
    best = max(fused, key=lambda k: (round(fused[k]["micro_f1"], 6), float(k.split("|")[1])))
    kind, alpha = best.split("|")[0], float(best.split("|")[1])
    dev = dict(fused[best], similarity=kind, alpha=alpha)
    print(f"[{name}] dev Micro-F1@7: model alone {grid['mean|model_only']['micro_f1']:.4f} (case vectors) / "
          f"{grid['maxsim|model_only']['micro_f1']:.4f} (best window pairs) | BM25 alone {BM25_DEV['micro_f1']:.4f} | "
          f"chosen: {kind} with alpha {alpha:.1f} -> {dev['micro_f1']:.4f} (P {dev['micro_precision']:.4f}, "
          f"R {dev['micro_recall']:.4f})", flush=True)
    ranked = top_k(fuse(neural[("test", kind)], BM25["test"], alpha))
    ranking_seconds = time.time() - t0
    zip_path = write_predictions(cfg, folder, ranked, dev["micro_f1"])
    result = {"name": name, "signature": signature, "config": {k: v for k, v in cfg.items() if k != "name"},
              "phase1_checkpoint": PHASE1[cfg["backbone"]]["dir"], "train": info, "dev": dev,
              "dev_grid": {k: v["micro_f1"] for k, v in grid.items()},
              "dev_model_only": {"mean": grid["mean|model_only"], "maxsim": grid["maxsim|model_only"]},
              "test_queries": len(ranked), "ranking_seconds": ranking_seconds, "submission": zip_path}
    write_json(result, results_path)
    print(f"[{name}] test: {len(ranked)} queries x {TOP_K} candidates -> {zip_path}\n", flush=True)
    release(model)
    RESULTS[name] = result
    return result

for _name in RUN_EXPERIMENTS:
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    try:
        run_experiment(_name)
    except Exception as _err:
        import traceback
        traceback.print_exc()
        print(f"[{_name}] FAILED: {_err} - continuing with the next experiment.\n")
        release()

## 11. Results and analysis

In [ ]:
finished = {name: r for name, r in RESULTS.items() if r is not None}
for _name in EXPERIMENTS:
    _path = os.path.join(WORK_ROOT, _name, "results.json")
    if _name not in finished and os.path.exists(_path):
        finished[_name] = load_json_file(_path)

rows = []
for name, r in sorted(finished.items(), key=lambda kv: -kv[1]["dev"]["micro_f1"]):
    cfg, d, t = r["config"], r["dev"], r["train"]
    rows.append({"experiment": name, "model": cfg["backbone"] + (f"[:{cfg['n_layers']}]" if cfg["n_layers"] else ""),
                 "technique": cfg["technique"] + (" + KD" if cfg["teacher"] else "") + (" cont." if cfg.get("init_from") else ""),
                 "dev_f1_before": t["history"][0]["dev_micro_f1"] if t["history"] and t["history"][0]["epoch"] == 0 else float("nan"),
                 "dev_f1_model_mean": r["dev_model_only"]["mean"]["micro_f1"], "dev_f1_model_maxsim": r["dev_model_only"]["maxsim"]["micro_f1"],
                 "dev_f1_bm25": BM25_DEV["micro_f1"], "dev_f1_final": d["micro_f1"], "dev_p": d["micro_precision"], "dev_r": d["micro_recall"],
                 "similarity": d["similarity"], "alpha": d["alpha"], "best_epoch": t["best_epoch"],
                 "params_M": t["parameters"]["total"] / 1e6, "trainable_M": t["parameters"]["trainable"] / 1e6,
                 "trainable_%": t["parameters"]["trainable_percent"], "train_min": t["train_seconds"] / 60,
                 "peak_mem_GB": t["peak_memory_mb"] / 1000, "submission": os.path.basename(r["submission"])})

if not rows:
    print("No finished experiments yet.")
else:
    header = (f"{'experiment':12s} {'model':6s} {'technique':11s} {'before':>7s} {'model':>7s} {'maxsim':>7s} {'BM25':>7s} "
              f"{'FINAL':>7s} {'P':>6s} {'R':>6s} {'sim':>6s} {'alpha':>5s} {'ep':>3s} {'params':>7s} {'trained':>8s} {'%':>6s} "
              f"{'train':>6s} {'mem':>5s}")
    print(f"Dev queries ({len(QUERIES['dev'])}), Micro-F1@7. before = Phase-1 model without fine-tuning; model / maxsim = the "
          f"fine-tuned model alone (case vectors / best window pairs); BM25 = lexical baseline alone; FINAL = the submitted "
          f"ranking (alpha x model + (1 - alpha) x BM25).\n" + header + "\n" + "-" * len(header))
    for x in rows:
        print(f"{x['experiment']:12s} {x['model']:6s} {x['technique']:11s} {x['dev_f1_before']:7.4f} {x['dev_f1_model_mean']:7.4f} "
              f"{x['dev_f1_model_maxsim']:7.4f} {x['dev_f1_bm25']:7.4f} {x['dev_f1_final']:7.4f} {x['dev_p']:6.3f} {x['dev_r']:6.3f} "
              f"{x['similarity']:>6s} {x['alpha']:5.1f} {x['best_epoch']:3d} {x['params_M']:6.1f}M {x['trainable_M']:7.3f}M "
              f"{x['trainable_%']:6.2f} {x['train_min']:5.1f}m {x['peak_mem_GB']:4.1f}G")
    with open(os.path.join(WORK_ROOT, "results_summary.csv"), "w", encoding="utf-8") as f:
        f.write(",".join(rows[0].keys()) + "\n")
        for x in rows:
            f.write(",".join(str(round(v, 5)) if isinstance(v, float) else str(v) for v in x.values()) + "\n")
    print("\nDev Micro-F1@7 for every BM25 weight (1 - alpha), best experiment first:")
    for x in rows:
        g = finished[x["experiment"]]["dev_grid"]
        print(f"  {x['experiment']:12s} mean:   " + " ".join(f"{a:.1f}:{g[f'mean|{a}']:.3f}" for a in ALPHA_GRID))
        print(f"  {'':12s} maxsim: " + " ".join(f"{a:.1f}:{g[f'maxsim|{a}']:.3f}" for a in ALPHA_GRID))
    print(f"\nSummary table saved to {os.path.join(WORK_ROOT, 'results_summary.csv')}")

## 12. Submission files

In [ ]:
if rows:
    print(f"Submission files in {SUBMISSION_DIR} (best dev score first):")
    for x in rows:
        print(f"  dev Micro-F1@7 {x['dev_f1_final']:.4f}   {x['submission']}  ({len(x['submission'])} characters)")
    with zipfile.ZipFile(os.path.join(SUBMISSION_DIR, rows[0]["submission"])) as z:
        _p = json.loads(z.read("prediction.json"))
    _k = next(iter(_p))
    print(f"\nprediction.json: {len(_p)} queries; first: {_k!r}: {_p[_k]}")

## 13. Round 3: citation- and date-aware re-ranking (two new submissions)

score = α·z(model) + (1−α)·z(lexical) + w_cite·log(1 + citation links) − w_date·[candidate decided after the query]. The lexical variant, w_cite and w_date are chosen on the 827 train queries; α and the similarity type on dev. A weight that does not help comes out 0.

In [ ]:
_YEAR = r"((?:19|20)\d\d)"
_CITE_PATTERNS = [
    ("AIR", re.compile(r"\bA\.?\s*I\.?\s*R\.?\s*\(?" + _YEAR + r"\)?\s*S\.?\s*C\.?\s*(\d{1,4})\b")),
    ("AIR", re.compile(r"\b" + _YEAR + r"\s*A\.?\s*I\.?\s*R\.?\s*(\d{1,4})\b")),
    ("AIR", re.compile(r"(?<![A-Z\d])" + _YEAR + r"\s+S\.?\s*C\.?\s+(\d{1,4})\b")),
    ("SCR", re.compile(r"\b" + _YEAR + r"\s*S\.?\s*C\.?\s*R\.?\s*\(\s*\d{1,2}\s*\)\s*(\d{1,4})\b")),
    ("SCR", re.compile(r"[\[(]" + _YEAR + r"[\])]\s*(?:\d{1,2}\s*)?S\.?\s*C\.?\s*R\.?\s*(\d{1,4})\b")),
    ("SCR", re.compile(r"\b" + _YEAR + r"\s*S\.?\s*C\.?\s*R\.?\s+(\d{1,4})\b")),
    ("SCC", re.compile(r"\(" + _YEAR + r"\)\s*\d{1,2}\s*S\.?\s*C\.?\s*C\.?\s*(\d{1,4})\b")),
    ("SCC", re.compile(r"\b" + _YEAR + r"\s*S\.?\s*C\.?\s*C\.?\s*\(\s*[\w.]{1,5}\s*\)\s*(\d{1,4})\b")),
]
_DATE = re.compile(r"DATE OF JUDGMENT\s*:?\s*(\d{1,2})\s*[/.\-]\s*(\d{1,2})\s*[/.\-]\s*((?:19|20)\d\d)")
_ENDS = ("ACT:", "ACT :", "HEADNOTE", "JUDGMENT:", "JUDGMENT :")

def citation_keys(text):
    keys = set()
    for reporter, pattern in _CITE_PATTERNS:
        for m in pattern.finditer(text):
            keys.add((reporter, int(m.group(1)), int(m.group(2))))
    return keys

def _segment(t, start_word, limit):
    i = t.find(start_word, 0, 4000)
    if i < 0:
        return None, None
    end = i + limit
    for e in _ENDS + (("CITATOR INFO",) if start_word != "CITATOR INFO" else ()):
        j = t.find(e, i + len(start_word))
        if 0 <= j < end:
            end = j
    return i, end

def case_info(text):
    """-> (own citation keys, keys in its CITATOR INFO, keys mentioned in the body, date as yyyymmdd or None)."""
    t = " ".join(str(text).upper().split())
    own, citator, cut = set(), set(), []
    i, e = _segment(t, "CITATION", 500)
    if i is not None:
        own = citation_keys(t[i:e])
        cut.append((i, e))
    else:
        own = citation_keys(t[:300])
    i, e = _segment(t, "CITATOR INFO", 2500)
    if i is not None:
        citator = citation_keys(t[i:e])
        cut.append((i, e))
    body = t
    for a, b in sorted(cut, reverse=True):
        body = body[:a] + " " + body[b:]
    m = _DATE.search(t[:4000])
    date = None
    if m:
        d, mo, y = int(m.group(1)), int(m.group(2)), int(m.group(3))
        if 1 <= mo <= 12 and 1 <= d <= 31:
            date = y * 10000 + mo * 100 + d
    return own, citator, citation_keys(body) - own, date

def header_features(split):
    """cite [queries x pool] (counts) and after [queries x pool] (bool), cached on disk."""
    path = os.path.join(CACHE_DIR, f"header_{split}_{len(QUERIES[split])}_{len(POOLS[split]['ids'])}.npz")
    if os.path.exists(path):
        z = np.load(path)
        return z["cite"], z["after"]
    cands = [case_info(t) for t in tqdm(POOLS[split]["texts"], desc=f"case headers ({split} pool)", leave=False)]
    queries = [case_info(q["text"]) for q in tqdm(QUERIES[split], desc=f"case headers ({split} queries)", leave=False)]
    by_own, by_citator = defaultdict(list), defaultdict(list)
    for j, (own, citator, _, _) in enumerate(cands):
        for k in own:
            by_own[k].append(j)
        for k in citator:
            by_citator[k].append(j)
    cite = np.zeros((len(queries), len(cands)), dtype=np.float32)
    after = np.zeros((len(queries), len(cands)), dtype=bool)
    c_dates = np.array([c[3] if c[3] is not None else -1 for c in cands])
    for i, (q_own, _, q_body, q_date) in enumerate(queries):
        for k in q_body:
            for j in by_own.get(k, ()):
                cite[i, j] += 1
        for k in q_own:
            for j in by_citator.get(k, ()):
                cite[i, j] += 1
        if q_date is not None:
            after[i] = (c_dates > q_date) & (c_dates > 0)
    np.savez(path, cite=cite, after=after)
    n_own = np.mean([len(c[0]) > 0 for c in cands])
    n_date = np.mean([c[3] is not None for c in cands])
    print(f"  {split}: candidates with their own citation found {100 * n_own:.0f}%, with a date {100 * n_date:.0f}%; "
          f"query-candidate pairs with a citation link: {int((cite > 0).sum()):,}")
    return cite, after

def bm25_top_terms(split, top_terms, k1=1.2, b=0.75):
    """BM25 where each (very long) query keeps only its top_terms rarest words (highest idf)."""
    path = os.path.join(CACHE_DIR, f"bm25top{top_terms}_{split}_{len(QUERIES[split])}_{len(POOLS[split]['ids'])}.npy")
    if os.path.exists(path):
        return np.load(path)
    vocab, rows, cols, vals, lengths = {}, [], [], [], []
    for i, t in enumerate(POOLS[split]["texts"]):
        c = Counter(_WORD.findall(t.lower()))
        ids = [vocab.setdefault(w, len(vocab)) for w in c]
        rows.append(np.full(len(c), i, dtype=np.int64))
        cols.append(np.asarray(ids, dtype=np.int64))
        vals.append(np.fromiter(c.values(), dtype=np.float32, count=len(c)))
        lengths.append(sum(c.values()))
    n_docs, V = len(lengths), len(vocab)
    tf = sp.coo_matrix((np.concatenate(vals), (np.concatenate(rows), np.concatenate(cols))), shape=(n_docs, V)).tocsr().tocoo()
    df = np.bincount(tf.col, minlength=V).astype(np.float32)
    idf = np.log(1.0 + (n_docs - df + 0.5) / (df + 0.5)).astype(np.float32)
    lengths = np.asarray(lengths, dtype=np.float32)
    norm = k1 * (1 - b + b * lengths / max(1.0, lengths.mean()))
    W = sp.csr_matrix((tf.data * (k1 + 1) / (tf.data + norm[tf.row]) * idf[tf.col], (tf.row, tf.col)), shape=(n_docs, V))
    qr, qc, qv = [], [], []
    for i, q in enumerate(QUERIES[split]):
        terms = [(vocab[w], n) for w, n in Counter(_WORD.findall(q["text"].lower())).items() if w in vocab]
        terms = sorted(terms, key=lambda x: -idf[x[0]])[:top_terms]
        for j, n in terms:
            qr.append(i); qc.append(j); qv.append(1.0 + math.log(n))
    Q = sp.csr_matrix((np.asarray(qv, dtype=np.float32), (qr, qc)), shape=(len(QUERIES[split]), V))
    s = (Q @ W.T).toarray().astype(np.float32)
    np.save(path, s)
    return s

def experiment_scores(name, split):
    """Model similarities (case vectors and best window pairs) from the saved embeddings of a finished experiment."""
    cfg, sig = experiment_config(name), RESULTS[name]["signature"]
    model = None
    out = {}
    emb = {}
    for kind in ("q", "c"):
        path = os.path.join(exp_dir(name), f"emb_{split}_{kind}_{MAX_CHUNKS}.npz")
        ok = os.path.exists(path) and str(np.load(path, allow_pickle=False)["signature"]) == sig
        if not ok and model is None:
            model = load_model(cfg)
        emb[kind] = cached_embeddings(model, cfg, sig, split, kind, MAX_CHUNKS)
    if model is not None:
        release(model)
    for sim in ("mean", "maxsim"):
        out[sim] = standardise(exclude_self(split, neural_scores(emb["q"], emb["c"], sim)))
    return out

print("Round 3: citation and date features from the case headers")
HEADER = {s: header_features(s) for s in ("train", "dev", "test")}
LEX = {s: {"bm25": standardise(BM25[s]),
           "bm25_top300": standardise(exclude_self(s, bm25_top_terms(s, 300)))} for s in ("train", "dev", "test")}

W_CITE, W_DATE = [0.0, 0.5, 1.0, 2.0, 4.0, 8.0], [0.0, 1.0, 1000.0]
best1 = None
for lex in LEX["train"]:
    for wc in W_CITE:
        for wd in W_DATE:
            s = LEX["train"][lex] + wc * np.log1p(HEADER["train"][0]) - wd * HEADER["train"][1]
            f = micro_prf("train", top_k(s))["micro_f1"]
            if best1 is None or f > best1[0] + 1e-9:
                best1 = (f, lex, wc, wd)
_, LEX_PICK, WC1, WD1 = best1
_gold_after = np.mean([HEADER["train"][1][i, j] for i, q in enumerate(QUERIES["train"]) for j in q["rel_idx"]])
_gold_cite = np.mean([HEADER["train"][0][i, j] > 0 for i, q in enumerate(QUERIES["train"]) for j in q["rel_idx"]])
print(f"  train: relevant cases with a citation link {100 * _gold_cite:.1f}%, decided after the query {100 * _gold_after:.1f}%")
for lex in LEX["train"]:
    print(f"  train Micro-F1@7, {lex} alone: {micro_prf('train', top_k(LEX['train'][lex]))['micro_f1']:.4f}")
print(f"  stage 1 choice: {LEX_PICK}, citation weight {WC1}, date penalty {WD1} -> train Micro-F1@7 {best1[0]:.4f}")

def round3(tag, members):
    """members: finished experiments whose standardised similarities are averaged (one = a single model)."""
    parts = {m: {s: experiment_scores(m, s) for s in ("dev", "test")} for m in members}
    best = None
    for sim in ("mean", "maxsim"):
        neural = {s: sum(parts[m][s][sim] for m in members) / len(members) for s in ("dev", "test")}
        neural = {s: np.where(np.isfinite(v), v, -np.inf) for s, v in neural.items()}
        for alpha in ALPHA_GRID:
            for wc in sorted({WC1 * 0.5, WC1, WC1 * 2.0}):
                def score(s, alpha=alpha, wc=wc, neural=neural):
                    return (alpha * np.where(np.isfinite(neural[s]), neural[s], 0.0)
                            + (1 - alpha) * np.where(np.isfinite(LEX[s][LEX_PICK]), LEX[s][LEX_PICK], 0.0)
                            + wc * np.log1p(HEADER[s][0]) - WD1 * HEADER[s][1])
                sd = score("dev")
                sd[~np.isfinite(neural["dev"])] = -np.inf
                m = micro_prf("dev", top_k(sd))
                if best is None or m["micro_f1"] > best[0]["micro_f1"] + 1e-9:
                    best = (m, sim, alpha, wc, score, neural)
    m, sim, alpha, wc, score, neural = best
    st = score("test")
    st[~np.isfinite(neural["test"])] = -np.inf
    ranked = top_k(st)
    print(f"[{tag}] {' + '.join(members)}: dev Micro-F1@7 {m['micro_f1']:.4f} (P {m['micro_precision']:.4f}, R {m['micro_recall']:.4f}) "
          f"with {sim}, alpha {alpha:.1f}, citation weight {wc}, date penalty {WD1}, lexical {LEX_PICK}", flush=True)
    folder = exp_dir(f"R3_{tag}")
    cfg = dict(experiment_config(members[0]), name=tag)
    zip_path = write_predictions(cfg, folder, ranked, m["micro_f1"])
    short = f"p7_{tag}_dev{m['micro_f1']:.4f}.zip"
    assert len(short) <= 64
    final = os.path.join(SUBMISSION_DIR, short)
    os.replace(zip_path, final)
    return final, m["micro_f1"]

FINAL_ZIPS = []
_single = [n for n in ("S1c_B_full", "S1_B_full") if RESULTS.get(n)]
_ens = [n for n in ("S1c_B_full", "S3c_A4_kd", "S2_D_lorma") if RESULTS.get(n)]
for _tag, _members in (("R3a_B-cite", _single[:1]), ("R3b_Ens3-cite", _ens)):
    try:
        if _members:
            FINAL_ZIPS.append(round3(_tag, _members))
    except Exception as _err:
        import traceback
        traceback.print_exc()
        print(f"[{_tag}] FAILED: {_err}")
OUT_DIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.path.dirname(WORK_ROOT)
print("\nThe two new submissions (also copied to the top of the output folder):")
for _z, _f in FINAL_ZIPS:
    shutil.copy2(_z, os.path.join(OUT_DIR, os.path.basename(_z)))
    print(f"  dev Micro-F1@7 {_f:.4f}   {os.path.basename(_z)}")